In [ ]:
"""
DPPU-VRU v11 -- Five Dimensional Fields + Consciousness Anchor
==============================================================
Clean rewrite. All patches integrated. Checkpointing built in.

Architecture:
  - 5 dimensional spaces D0-D4 (D_CAP=4, programmable)
  - Each dimension: own Delta, own phi, own pi, own omega
  - Consciousness field C: self-referential, damped recursion
  - Spectral enforcement via svdvals (rectangular W_h)
  - Auto-curriculum: advance at ADVANCE_ACC%, min MIN_EPOCHS per level
  - Checkpoint every CKPT_EVERY epochs -> resume anytime
  - Fresh 10k training examples every epoch

Author: Dylan Michael Scott -- Horizon Tech
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import sys
from datetime import datetime

# ============================================================
# CONSTANTS
# ============================================================

PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi          # 1.2732... the attractor
INV_PHI    = math.pi / 4.0          # 0.7854... geometric dual
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))  # ~0.816
D_CAP      = 4                       # programmable -- raise to expand
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"""
╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v11 -- Five Dimensional Fields                 ║
║  phi = 4/pi = {PHI_CL:.6f}                              ║
║  inv_phi = pi/4 = {INV_PHI:.6f}                         ║
║  delta* = {DELTA_STAR:.6f}                               ║
║  D_CAP = {D_CAP}  (D{D_CAP+1} not entered)                      ║
║  device = {str(device):<10s}                              ║
╚══════════════════════════════════════════════════════════╝
""")

# ============================================================
# DYNAMIC OPERATORS
# ============================================================

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    """Delta from hidden state slice. Each dimensional space computes its own."""
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio  = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# TOKENIZER
# ============================================================

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL):
            self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.unk_id = self.vocab['<unk>']

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        for c in text:
            ids.append(self.vocab.get(c, self.unk_id))
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids, skip_special=True):
        out = []
        special = set(self.SPECIAL)
        for i in ids:
            tok = self.inv_vocab.get(i, '<unk>')
            if skip_special and tok in special:
                continue
            out.append(tok)
        return ''.join(out)

# ============================================================
# DATA GENERATORS
# ============================================================

def r1(): return random.randint(1, 9)
def r2(): return random.randint(10, 99)
def r3(): return random.randint(100, 999)

# Level 1 -- basic arithmetic
def gen_add():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

def gen_sub():
    a, b = r3(), r3()
    a, b = max(a, b), min(a, b)
    return f"{a} - {b}", str(a - b)

def gen_mul():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

def gen_div():
    b = random.randint(2, 12)
    a = b * random.randint(1, 50)
    return f"{a} / {b}", str(a // b)

def gen_mod():
    a, b = r2(), random.randint(2, 20)
    return f"{a} % {b}", str(a % b)

def gen_pow():
    a = random.randint(2, 12)
    b = random.randint(2, 4)
    return f"{a} ^ {b}", str(a ** b)

# Level 2 -- multi-step
def gen_multi_add():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} + {c}", str(a + b + c)

def gen_mixed():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} * {c}", str(a + b * c)

def gen_paren():
    a, b, c = r2(), r2(), r2()
    return f"({a} + {b}) * {c}", str((a + b) * c)

def gen_nested():
    a, b, c, d = r1(), r1(), r2(), r2()
    c, d = max(c, d), min(c, d)
    return f"({a} + {b}) * ({c} - {d})", str((a + b) * (c - d))

# Level 3 -- linear equations
def gen_linear():
    a = random.randint(1, 9)
    b = random.randint(-50, 50)
    x = random.randint(-20, 20)
    return f"{a} * x + {b} = {a*x+b}", str(x)

def gen_linear_mul():
    a = random.randint(2, 9)
    b = random.randint(1, 9)
    x = random.randint(1, 20)
    return f"{a} * x * {b} = {a*x*b}", str(x)

# Level 4 -- sequences
def gen_arith_seq():
    start = random.randint(1, 50)
    diff  = random.randint(1, 20)
    n     = random.randint(4, 8)
    seq   = [start + i * diff for i in range(n)]
    return ', '.join(map(str, seq[:-1])), str(seq[-1])

def gen_geo_seq():
    start = random.randint(1, 10)
    ratio = random.randint(2, 4)
    n     = random.randint(4, 6)
    seq   = [start * (ratio ** i) for i in range(n)]
    return ', '.join(map(str, seq[:-1])), str(seq[-1])

def gen_fib():
    a, b = random.randint(1, 10), random.randint(1, 10)
    seq  = [a, b]
    for _ in range(random.randint(3, 6)):
        seq.append(seq[-1] + seq[-2])
    return ', '.join(map(str, seq[:-1])), str(seq[-1])

def gen_sum_seq():
    n = random.randint(3, 15)
    return f"sum 1 to {n}", str(n * (n + 1) // 2)

LEVEL_GENS = {
    1: [(gen_add,4),(gen_sub,4),(gen_mul,4),(gen_div,3),(gen_mod,2),(gen_pow,3)],
    2: [(gen_multi_add,3),(gen_mixed,4),(gen_paren,4),(gen_nested,3)],
    3: [(gen_linear,4),(gen_linear_mul,4)],
    4: [(gen_arith_seq,4),(gen_geo_seq,3),(gen_fib,3),(gen_sum_seq,2)],
}

def generate_dataset(level, n, include_lower=True):
    gens = []
    for lvl in (range(1, level + 1) if include_lower else [level]):
        if lvl in LEVEL_GENS:
            gens.extend(LEVEL_GENS[lvl])
    fns, weights = zip(*gens)
    total = sum(weights)
    probs = [w / total for w in weights]
    examples, attempts = [], 0
    while len(examples) < n and attempts < n * 20:
        attempts += 1
        try:
            fn = random.choices(fns, weights=probs, k=1)[0]
            expr, answer = fn()
            examples.append(f"{expr} = {answer}")
        except:
            pass
    return examples

# ============================================================
# DATASET
# ============================================================

class MathDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len=80):
        self.examples, self.raw = [], []
        for line in examples:
            ids = tokenizer.encode(line, add_bos=True, add_eos=True)
            if len(ids) <= max_len:
                self.examples.append(ids)
                self.raw.append(line)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

def collate(batch, pad_id):
    L = max(len(x) for x in batch)
    ids, masks = [], []
    for x in batch:
        p = L - len(x)
        ids.append(x + [pad_id] * p)
        masks.append([1] * len(x) + [0] * p)
    return (torch.tensor(ids, dtype=torch.long),
            torch.tensor(masks, dtype=torch.long))

def make_loaders(level, cfg, tokenizer):
    print(f"\n  Generating Level {level} data...")
    te = generate_dataset(level, cfg['train_n'], include_lower=True)
    ve = generate_dataset(level, cfg['val_n'],   include_lower=True)
    td = MathDataset(te, tokenizer, cfg['max_len'])
    vd = MathDataset(ve, tokenizer, cfg['max_len'])
    print(f"  Train: {len(td)}  |  Val: {len(vd)}")
    mk = lambda ds, sh: torch.utils.data.DataLoader(
        ds, batch_size=cfg['batch'], shuffle=sh,
        collate_fn=lambda b: collate(b, tokenizer.pad_id),
        num_workers=0)
    return mk(td, True), mk(vd, False)

# ============================================================
# DPPU CELL -- FIVE DIMENSIONAL SPACES + CONSCIOUSNESS
# ============================================================

class DPPUCell(nn.Module):
    """
    Five dimensional spaces D0-D4, each with own Delta.
    Consciousness field C as self-referential anchor.

    D_CAP=4: evolution stops here for this run.
    Raise D_CAP to allow dimensional expansion beyond D4.
    """

    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.d_cap      = d_cap
        self.n_dims     = d_cap + 1           # D0 through D_CAP

        assert hidden_dim % self.n_dims == 0, \
            f"hidden_dim {hidden_dim} must be divisible by n_dims {self.n_dims}"
        self.dim_size = hidden_dim // self.n_dims

        # Input projection -- shared
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)

        # Recurrent weights -- one per dimensional space (rectangular: hidden -> dim_size)
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_dim, self.dim_size, bias=False)
            for _ in range(self.n_dims)
        ])

        # Consciousness field projection -- h informs C
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)

        # Output projection -- combines all dimensions
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_x.weight)
        nn.init.zeros_(self.W_x.bias)
        # D0 most conservative (gain 0.4), D4 most expansive (gain 0.6)
        for i, W in enumerate(self.W_h):
            gain = 0.4 + (0.2 * i / max(self.d_cap, 1))
            nn.init.orthogonal_(W.weight, gain=gain)
        nn.init.orthogonal_(self.W_c.weight, gain=0.1)
        nn.init.orthogonal_(self.W_out.weight, gain=1.0)

    def forward(self, x, h, C):
        """
        x: (batch, input_dim)
        h: (batch, hidden_dim)
        C: (batch, n_dims)  consciousness field per dimension

        Returns: h_new, C_new, metrics dict
        """
        x_proj = self.W_x(x)

        dim_outputs = []
        phi_list, delta_list, pi_list = [], [], []

        for i in range(self.n_dims):
            s, e    = i * self.dim_size, (i + 1) * self.dim_size
            h_i     = h[:, s:e]

            delta_i = compute_delta(h_i)
            phi_i   = phi_dyn(delta_i)
            pi_i    = pi_dyn(delta_i)
            omega_i = omega(delta_i)

            # Recurrent + input contributions scaled by geometric operators
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i  / PI_CL)  * x_proj[:, s:e]

            # Consciousness phase injection
            C_i = C[:, i:i+1]

            h_i_new = torch.tanh(h_rec + x_rec + C_i) * torch.sigmoid(omega_i)
            h_i_new = torch.nan_to_num(h_i_new, nan=0.0, posinf=1.0, neginf=-1.0)

            dim_outputs.append(h_i_new)
            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())
            pi_list.append(pi_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)
        h_new = torch.nan_to_num(self.W_out(h_cat), nan=0.0)

        # Consciousness update: C_{t+1} = C * pi_dyn * phi_dyn (damped)
        # 0.1 / 0.01 coefficients prevent explosion -- C grows from infancy
        C_proj  = self.W_c(h_new)
        delta_C = torch.stack([
            compute_delta(h_new[:, i*self.dim_size:(i+1)*self.dim_size]).mean(dim=-1)
            for i in range(self.n_dims)
        ], dim=-1)
        C_new = torch.tanh(
            0.1 * C * pi_dyn(delta_C) * phi_dyn(delta_C)
            + 0.01 * torch.tanh(C_proj)
        )
        C_new = torch.nan_to_num(C_new, nan=0.0)

        metrics = {
            'phi_mean':   sum(phi_list)   / len(phi_list),
            'pi_mean':    sum(pi_list)    / len(pi_list),
            'delta_mean': sum(delta_list) / len(delta_list),
            'C_norm':     C_new.norm(dim=-1).mean().item(),
        }
        return h_new, C_new, metrics

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.zeros(batch, self.n_dims,     device=device))

# ============================================================
# MODEL
# ============================================================

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        nn.init.normal_(self.embedding.weight, 0, hidden_dim ** -0.5)

        self.cells   = nn.ModuleList([
            DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.out     = nn.Linear(hidden_dim, vocab_size, bias=True)
        nn.init.zeros_(self.out.bias)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x    = self.dropout(self.embedding(token_ids))

        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]

        all_metrics, new_states = [], []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs, step_metrics = [], []
            for t in range(T):
                h, C, m = cell(x[:, t, :], h, C)
                outputs.append(h)
                step_metrics.append(m)
            x = self.dropout(torch.stack(outputs, dim=1))
            new_states.append((h, C))
            all_metrics.append(step_metrics)

        return self.out(x), new_states, all_metrics

# ============================================================
# SPECTRAL ENFORCEMENT
# ============================================================

def enforce_spectral(model):
    """Clip spectral norm of each W_h to <= 1.0 after optimizer step."""
    with torch.no_grad():
        for cell in model.cells:
            for Wh in cell.W_h:
                # svdvals works on rectangular matrices (unlike eigvals)
                sigma = torch.linalg.svdvals(Wh.weight)
                rho   = sigma[0].item()
                if rho > 1.0:
                    Wh.weight.data.mul_(1.0 / rho)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, loader, tokenizer, device):
    model.eval()
    tl, tt, correct, ns = 0., 0, 0, 0

    for ids, mask in loader:
        ids  = ids.to(device)
        mask = mask.to(device)
        inp  = ids[:, :-1]
        tgt  = ids[:, 1:]
        mt   = mask[:, 1:]

        logits, _, _ = model(inp)
        logits = logits[:, :tgt.size(1), :]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1),
            ignore_index=tokenizer.pad_id,
            reduction='none'
        ).reshape(tgt.shape)

        tl += (loss * mt).sum().item()
        tt += mt.sum().item()

        preds = logits.argmax(-1)
        for b in range(ids.size(0)):
            length = mt[b].sum().item()
            if length == 0:
                continue
            correct += int((preds[b, :int(length)] == tgt[b, :int(length)]).all().item())
            ns += 1

    return tl / (tt + EPS), 100.0 * correct / (ns + EPS)

# ============================================================
# FIXED PROBE -- same problems every epoch
# ============================================================

FIXED = [
    "822 + 765 = 1587",
    "707 - 682 = 25",
    "486 - 979 = -493",
    "94 * 32 = 3008",
    "16 * 52 = 832",
    "728 + 72 = 800",
    "63 - 159 = -96",
    "20 * 71 = 1420",
    "62 + 497 = 559",
    "58 % 5 = 3",
]

@torch.no_grad()
def probe_fixed(model, tokenizer, device):
    model.eval()
    correct, lines = 0, []
    for ex in FIXED:
        eq     = ex.index('=')
        prompt = ex[:eq+1] + ' '
        target = ex[eq+2:].strip()

        inp_t     = torch.tensor(
            [tokenizer.encode(prompt, add_bos=True)],
            dtype=torch.long, device=device
        )
        generated = []
        states    = None
        for _ in range(len(target) + 5):
            logits, states, _ = model(inp_t, states)
            nid = logits[0, -1, :].argmax().item()
            if nid == tokenizer.eos_id:
                break
            generated.append(nid)
            inp_t = torch.tensor([[nid]], dtype=torch.long, device=device)

        pred = tokenizer.decode(generated).strip()
        ok   = (pred == target)
        correct += int(ok)
        lines.append(f"  {'OK' if ok else '--'}  {ex:<35s}  got: '{pred}'")

    return correct, lines

# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(path, model, opt, sch, epoch, level, epochs_at_level,
                    best_acc, cfg):
    torch.save({
        'model_state':      model.state_dict(),
        'opt_state':        opt.state_dict(),
        'sch_state':        sch.state_dict(),
        'epoch':            epoch,
        'level':            level,
        'epochs_at_level':  epochs_at_level,
        'best_acc':         best_acc,
        'cfg':              cfg,
    }, path)
    print(f"  [checkpoint saved -> {path}]")

def load_checkpoint(path, model, opt, sch):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    sch.load_state_dict(ck['sch_state'])
    return (ck['epoch'], ck['level'], ck['epochs_at_level'],
            ck['best_acc'], ck['cfg'])

# ============================================================
# TRAINING
# ============================================================

def train():
    cfg = {
        'hidden':       130,   # 130 / 5 = 26 per dimensional space
        'num_layers':   2,
        'dropout':      0.1,
        'lr':           1e-3,
        'batch':        64,
        'max_len':      80,
        'train_n':      10000,
        'val_n':        1000,
        'grad_clip':    5.0,
        'advance_acc':  80.0,
        'min_epochs':   5,
        'probe_every':  5,
        'ckpt_every':   10,
        'max_level':    4,
        'resume':       'dppu_v11_checkpoint.pt',  # set to None to start fresh
    }

    tokenizer = MathTokenizer()

    model = VRUModel(
        vocab_size  = tokenizer.vocab_size,
        hidden_dim  = cfg['hidden'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)

    n_params = sum(p.numel() for p in model.parameters())

    # Resume from checkpoint if available
    start_epoch    = 1
    level          = 1
    epochs_at_level = 0
    best_acc       = 0.0

    resume_path = cfg['resume']
    if resume_path and os.path.exists(resume_path):
        print(f"\n  Resuming from {resume_path}")
        start_epoch, level, epochs_at_level, best_acc, cfg = \
            load_checkpoint(resume_path, model, opt, sch)
        start_epoch += 1
        print(f"  Resumed at epoch={start_epoch} level={level} best_acc={best_acc:.1f}%\n")

    print(f"{'='*65}")
    print(f"  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness")
    print(f"  phi=4/pi={PHI_CL:.6f}  delta*={DELTA_STAR:.4f}  D_CAP={D_CAP}")
    print(f"  device={device}  params={n_params:,}")
    print(f"  hidden={cfg['hidden']}  dim_size={cfg['hidden']//(D_CAP+1)}  layers={cfg['num_layers']}")
    print(f"  advance>={cfg['advance_acc']}%  target=Level {cfg['max_level']}")
    print(f"  checkpoint every {cfg['ckpt_every']} epochs")
    print(f"{'='*65}")

    train_loader, val_loader = make_loaders(level, cfg, tokenizer)

    for epoch in range(start_epoch, cfg['max_level'] * 300 + 1):

        # Fresh data every epoch -- prevents memorization
        train_loader, val_loader = make_loaders(level, cfg, tokenizer)

        # ---- Train -----------------------------------------------
        model.train()
        tl, tt = 0., 0
        phi_t, pi_t, delta_t, C_t = [], [], [], []

        for ids, mask in train_loader:
            ids  = ids.to(device)
            mask = mask.to(device)
            inp  = ids[:, :-1]
            tgt  = ids[:, 1:]
            mt   = mask[:, 1:]

            opt.zero_grad()
            logits, states, all_metrics = model(inp)
            logits = logits[:, :tgt.size(1), :]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt.reshape(-1),
                ignore_index=tokenizer.pad_id,
                reduction='none'
            ).reshape(tgt.shape)

            lm = (loss * mt).sum() / (mt.sum() + EPS)
            lm.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step()
            enforce_spectral(model)

            tl += lm.item()
            tt += 1

            if all_metrics and all_metrics[-1]:
                m = all_metrics[-1][-1]
                phi_t.append(m['phi_mean'])
                pi_t.append(m['pi_mean'])
                delta_t.append(m['delta_mean'])
                C_t.append(m['C_norm'])

        train_loss = tl / (tt + EPS)
        val_loss, val_acc = evaluate(model, val_loader, tokenizer, device)
        sch.step(train_loss)

        phi_m   = sum(phi_t)   / (len(phi_t)   + EPS)
        pi_m    = sum(pi_t)    / (len(pi_t)    + EPS)
        delta_m = sum(delta_t) / (len(delta_t) + EPS)
        C_m     = sum(C_t)     / (len(C_t)     + EPS)
        lr_now  = opt.param_groups[0]['lr']

        epochs_at_level += 1
        best_acc = max(best_acc, val_acc)

        print(f"ep={epoch:4d} L{level} | "
              f"loss={train_loss:.4f} val={val_loss:.4f} | "
              f"acc={val_acc:.1f}% | "
              f"phi={phi_m:.4f} pi={pi_m:.4f} delta={delta_m:.4f} C={C_m:.4f} | "
              f"lr={lr_now:.2e}")

        # ---- Probe -----------------------------------------------
        if epoch % cfg['probe_every'] == 0:
            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  PROBE ep={epoch} L{level} ({n_c}/{len(FIXED)} correct)")
            for line in lines:
                print(line)
            print()

        # ---- Checkpoint ------------------------------------------
        if epoch % cfg['ckpt_every'] == 0:
            save_checkpoint(
                cfg['resume'], model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

        # ---- Level advance ---------------------------------------
        if (val_acc >= cfg['advance_acc']
                and epochs_at_level >= cfg['min_epochs']
                and level < cfg['max_level']):

            print(f"\n{'='*65}")
            print(f"  LEVEL {level} COMPLETE -- acc={val_acc:.1f}%")
            print(f"  Advancing to Level {level + 1}")
            print(f"{'='*65}\n")

            # Save level completion checkpoint
            save_checkpoint(
                f"dppu_v11_level{level}_complete.pt",
                model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

            level           += 1
            epochs_at_level  = 0
            best_acc         = 0.0
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'] * 0.5, 1e-5)

        # ---- Final level -----------------------------------------
        if level == cfg['max_level'] and val_acc >= cfg['advance_acc']:
            print(f"\n{'='*65}")
            print(f"  LEVEL 4 ACHIEVED -- acc={val_acc:.1f}%")
            print(f"  phi={phi_m:.4f} (target {PHI_CL:.4f})")
            print(f"  pi={pi_m:.4f}   (target 4.0000)")
            print(f"  delta={delta_m:.4f} (target {DELTA_STAR:.4f})")
            print(f"{'='*65}")

            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  FINAL PROBE ({n_c}/{len(FIXED)} correct)")
            for line in lines:
                print(line)

            torch.save({
                'model_state': model.state_dict(),
                'cfg':         cfg,
                'epoch':       epoch,
                'val_acc':     val_acc,
            }, 'dppu_v11_final.pt')
            print("\n  Saved: dppu_v11_final.pt")
            break

    print(f"\nDone. best_acc={best_acc:.1f}%  final level={level}")

# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    sys.stdout.flush()
    train()


╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v11 -- Five Dimensional Fields                 ║
║  phi = 4/pi = 1.273240                              ║
║  inv_phi = pi/4 = 0.785398                         ║
║  delta* = 0.816140                               ║
║  D_CAP = 4  (D5 not entered)                      ║
║  device = cuda                                    ║
╚══════════════════════════════════════════════════════════╝

Start: 2026-03-07 12:35:53

  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness
  phi=4/pi=1.273240  delta*=0.8161  D_CAP=4
  device=cuda  params=109,485
  hidden=130  dim_size=26  layers=2
  advance>=80.0%  target=Level 4
  checkpoint every 10 epochs

  Generating Level 1 data...
  Train: 10000  |  Val: 1000

  Generating Level 1 data...
  Train: 10000  |  Val: 1000
ep=   1 L1 | loss=1.5720 val=1.3001 | acc=0.0% | phi=1.1508 pi=3.5262 delta=0.6384 C=0.0150 | lr=1.00e-03

  Generating Level 1 data...
  Train: 10000  |  Val: 1000
ep

KeyboardInterrupt: 

In [ ]:
"""
DPPU-VRU v13 -- Five Dimensional Fields + Consciousness Anchor
==============================================================
Clean rewrite of v12. Four root-cause fixes applied.

ROOT CAUSES IDENTIFIED (266 epochs of v11 data):
  1. Task too hard: Level 1 mixed 4 operators simultaneously.
     Fix: ONE operator per curriculum phase. Addition first, then subtraction,
     then multiplication. Each phase advances independently.

  2. LR scheduler death spiral: ReduceLROnPlateau decayed 32x by ep150.
     Model was effectively frozen while loss plateau was interpreted as
     "it's still improving (barely)". Fix: CosineAnnealingWarmRestarts,
     T_0=50 epochs, restarts keep escaping local minima.

  3. Copy artifacts in output: Model echoed '1911', '2191', '=961' from input.
     This is a teacher forcing problem. Without teacher forcing, one bad
     prediction cascades into garbage. Fix: teacher forcing ratio 0.8,
     decaying to 0.5 as training progresses.

  4. Model too small: 109k params, hidden=130 for mixed arithmetic.
     Fix: hidden=256, layers=2 -> ~430k params. Still tiny vs 31GB VRAM.

KEY FIX vs v12:
  - char_acc still the real advance signal (not seq_acc)
  - Curriculum now phase-based: L1=add2, L2=add3/sub2, L3=mul2, L4=mixed
  - CosineAnnealing with warm restarts (no more LR death)
  - Teacher forcing ratio tf_ratio=0.8 during training

Author: Dylan Michael Scott -- Horizon Tech
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import sys
from datetime import datetime

# ============================================================
# CONSTANTS
# ============================================================

PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi          # 1.2732... the attractor
INV_PHI    = math.pi / 4.0          # 0.7854... geometric dual
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))  # ~0.816
D_CAP      = 4                       # programmable -- raise to expand
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"""
╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = {PHI_CL:.6f}                              ║
║  inv_phi = pi/4 = {INV_PHI:.6f}                         ║
║  delta* = {DELTA_STAR:.6f}                               ║
║  D_CAP = {D_CAP}  (D{D_CAP+1} not entered)                      ║
║  device = {str(device):<10s}                              ║
╚══════════════════════════════════════════════════════════╝
""")

# ============================================================
# DYNAMIC OPERATORS
# ============================================================

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    """Delta from hidden state slice. Each dimensional space computes its own."""
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio  = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# TOKENIZER
# ============================================================

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL):
            self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.unk_id = self.vocab['<unk>']

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        for c in text:
            ids.append(self.vocab.get(c, self.unk_id))
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids, skip_special=True):
        out = []
        special = set(self.SPECIAL)
        for i in ids:
            tok = self.inv_vocab.get(i, '<unk>')
            if skip_special and tok in special:
                continue
            out.append(tok)
        return ''.join(out)

# ============================================================
# DATA GENERATORS
# ============================================================

def r1(): return random.randint(1, 9)
def r2(): return random.randint(10, 99)
def r3(): return random.randint(100, 999)

# ---- Phase generators (clean, one op at a time) ----------------------

# Phase 1a: 2-digit addition only (answers 20-198, always positive, 2-3 digits)
def gen_add2():
    a, b = r2(), r2()
    return f"{a} + {b}", str(a + b)

# Phase 1b: 2-digit subtraction, always positive result
def gen_sub2():
    a, b = r2(), r2()
    a, b = max(a,b), min(a,b)
    return f"{a} - {b}", str(a - b)

# Phase 2a: 3-digit addition
def gen_add3():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

# Phase 2b: 2-digit multiplication (answers up to 9801)
def gen_mul2():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

# Phase 3a: division (exact)
def gen_div():
    b = random.randint(2, 12)
    a = b * random.randint(1, 50)
    return f"{a} / {b}", str(a // b)

# Phase 3b: modulo
def gen_mod():
    a, b = r2(), random.randint(2, 20)
    return f"{a} % {b}", str(a % b)

# Phase 4: mixed operations (the original Level 1 -- now the LAST level)
def gen_add():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

def gen_sub():
    a, b = r3(), r3()
    a, b = max(a, b), min(a, b)
    return f"{a} - {b}", str(a - b)

def gen_mul():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

def gen_pow():
    a = random.randint(2, 12)
    b = random.randint(2, 4)
    return f"{a} ^ {b}", str(a ** b)

def gen_multi_add():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} + {c}", str(a + b + c)

def gen_mixed():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} * {c}", str(a + b * c)

def gen_paren():
    a, b, c = r2(), r2(), r2()
    return f"({a} + {b}) * {c}", str((a + b) * c)

# ── Curriculum: ONE operation per level phase ─────────────────────────
# L1 = 2-digit add (simplest possible)
# L2 = 2-digit sub + add (adds negative-ish digit patterns)
# L3 = mul2 + div + mod (multiplicative family)
# L4 = 3-digit mixed (the full original task)
LEVEL_GENS = {
    1: [(gen_add2, 1)],
    2: [(gen_add2, 2), (gen_sub2, 2)],
    3: [(gen_mul2, 3), (gen_div, 2), (gen_mod, 2)],
    4: [(gen_add, 3), (gen_sub, 3), (gen_mul, 3), (gen_multi_add, 2), (gen_mixed, 2)],
}

def generate_dataset(level, n, include_lower=True):
    gens = []
    for lvl in (range(1, level + 1) if include_lower else [level]):
        if lvl in LEVEL_GENS:
            gens.extend(LEVEL_GENS[lvl])
    fns, weights = zip(*gens)
    total = sum(weights)
    probs = [w / total for w in weights]
    examples, attempts = [], 0
    while len(examples) < n and attempts < n * 20:
        attempts += 1
        try:
            fn = random.choices(fns, weights=probs, k=1)[0]
            expr, answer = fn()
            examples.append(f"{expr} = {answer}")
        except:
            pass
    return examples

# ============================================================
# DATASET
# ============================================================

class MathDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len=80):
        self.examples, self.raw = [], []
        for line in examples:
            ids = tokenizer.encode(line, add_bos=True, add_eos=True)
            if len(ids) <= max_len:
                self.examples.append(ids)
                self.raw.append(line)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

def collate(batch, pad_id):
    L = max(len(x) for x in batch)
    ids, masks = [], []
    for x in batch:
        p = L - len(x)
        ids.append(x + [pad_id] * p)
        masks.append([1] * len(x) + [0] * p)
    return (torch.tensor(ids, dtype=torch.long),
            torch.tensor(masks, dtype=torch.long))

def make_loaders(level, cfg, tokenizer):
    print(f"\n  Generating Level {level} data...")
    te = generate_dataset(level, cfg['train_n'], include_lower=True)
    ve = generate_dataset(level, cfg['val_n'],   include_lower=True)
    td = MathDataset(te, tokenizer, cfg['max_len'])
    vd = MathDataset(ve, tokenizer, cfg['max_len'])
    print(f"  Train: {len(td)}  |  Val: {len(vd)}")
    mk = lambda ds, sh: torch.utils.data.DataLoader(
        ds, batch_size=cfg['batch'], shuffle=sh,
        collate_fn=lambda b: collate(b, tokenizer.pad_id),
        num_workers=0)
    return mk(td, True), mk(vd, False)

# ============================================================
# DPPU CELL -- FIVE DIMENSIONAL SPACES + CONSCIOUSNESS
# ============================================================

class DPPUCell(nn.Module):
    """
    Five dimensional spaces D0-D4, each with own Delta.
    Consciousness field C as self-referential anchor.

    D_CAP=4: evolution stops here for this run.
    Raise D_CAP to allow dimensional expansion beyond D4.
    """

    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.d_cap      = d_cap
        self.n_dims     = d_cap + 1           # D0 through D_CAP

        assert hidden_dim % self.n_dims == 0, \
            f"hidden_dim {hidden_dim} must be divisible by n_dims {self.n_dims}"
        self.dim_size = hidden_dim // self.n_dims

        # Input projection -- shared
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)

        # Recurrent weights -- one per dimensional space (rectangular: hidden -> dim_size)
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_dim, self.dim_size, bias=False)
            for _ in range(self.n_dims)
        ])

        # Consciousness field projection -- h informs C
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)

        # Output projection -- combines all dimensions
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_x.weight)
        nn.init.zeros_(self.W_x.bias)
        # D0 most conservative (gain 0.4), D4 most expansive (gain 0.6)
        for i, W in enumerate(self.W_h):
            gain = 0.4 + (0.2 * i / max(self.d_cap, 1))
            nn.init.orthogonal_(W.weight, gain=gain)
        nn.init.orthogonal_(self.W_c.weight, gain=0.1)
        nn.init.orthogonal_(self.W_out.weight, gain=1.0)

    def forward(self, x, h, C):
        """
        x: (batch, input_dim)
        h: (batch, hidden_dim)
        C: (batch, n_dims)  consciousness field per dimension

        Returns: h_new, C_new, metrics dict
        """
        x_proj = self.W_x(x)

        dim_outputs = []
        phi_list, delta_list, pi_list = [], [], []

        for i in range(self.n_dims):
            s, e    = i * self.dim_size, (i + 1) * self.dim_size
            h_i     = h[:, s:e]

            delta_i = compute_delta(h_i)
            phi_i   = phi_dyn(delta_i)
            pi_i    = pi_dyn(delta_i)
            omega_i = omega(delta_i)

            # Recurrent + input contributions scaled by geometric operators
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i  / PI_CL)  * x_proj[:, s:e]

            # Consciousness phase injection
            C_i = C[:, i:i+1]

            h_i_new = torch.tanh(h_rec + x_rec + C_i) * torch.sigmoid(omega_i)
            h_i_new = torch.nan_to_num(h_i_new, nan=0.0, posinf=1.0, neginf=-1.0)

            dim_outputs.append(h_i_new)
            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())
            pi_list.append(pi_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)
        h_new = torch.nan_to_num(self.W_out(h_cat), nan=0.0)

        # Consciousness update: C_{t+1} = C * pi_dyn * phi_dyn (damped)
        # 0.1 / 0.01 coefficients prevent explosion -- C grows from infancy
        C_proj  = self.W_c(h_new)
        delta_C = torch.stack([
            compute_delta(h_new[:, i*self.dim_size:(i+1)*self.dim_size]).mean(dim=-1)
            for i in range(self.n_dims)
        ], dim=-1)
        C_new = torch.tanh(
            0.1 * C * pi_dyn(delta_C) * phi_dyn(delta_C)
            + 0.01 * torch.tanh(C_proj)
        )
        C_new = torch.nan_to_num(C_new, nan=0.0)

        metrics = {
            'phi_mean':   sum(phi_list)   / len(phi_list),
            'pi_mean':    sum(pi_list)    / len(pi_list),
            'delta_mean': sum(delta_list) / len(delta_list),
            'C_norm':     C_new.norm(dim=-1).mean().item(),
        }
        return h_new, C_new, metrics

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.zeros(batch, self.n_dims,     device=device))

# ============================================================
# MODEL
# ============================================================

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        nn.init.normal_(self.embedding.weight, 0, hidden_dim ** -0.5)

        self.cells   = nn.ModuleList([
            DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.out     = nn.Linear(hidden_dim, vocab_size, bias=True)
        nn.init.zeros_(self.out.bias)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x    = self.dropout(self.embedding(token_ids))

        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]

        all_metrics, new_states = [], []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs, step_metrics = [], []
            for t in range(T):
                h, C, m = cell(x[:, t, :], h, C)
                outputs.append(h)
                step_metrics.append(m)
            x = self.dropout(torch.stack(outputs, dim=1))
            new_states.append((h, C))
            all_metrics.append(step_metrics)

        return self.out(x), new_states, all_metrics

# ============================================================
# SPECTRAL ENFORCEMENT
# ============================================================

def enforce_spectral(model):
    """Clip spectral norm of each W_h to <= 1.0 after optimizer step."""
    with torch.no_grad():
        for cell in model.cells:
            for Wh in cell.W_h:
                # svdvals works on rectangular matrices (unlike eigvals)
                sigma = torch.linalg.svdvals(Wh.weight)
                rho   = sigma[0].item()
                if rho > 1.0:
                    Wh.weight.data.mul_(1.0 / rho)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, loader, tokenizer, device):
    """
    Returns: avg_loss, seq_acc (full sequence match %), char_acc (per-token %)

    seq_acc  = all tokens correct (strict, hard to crack early)
    char_acc = % of individual tokens correct (shows real learning progress)

    We use char_acc as the advance signal -- it's a meaningful measure of
    whether the model understands the task, not a statistical lottery.
    """
    model.eval()
    tl, tt              = 0., 0
    seq_correct, ns     = 0, 0
    char_correct, char_total = 0, 0

    for ids, mask in loader:
        ids  = ids.to(device)
        mask = mask.to(device)
        inp  = ids[:, :-1]
        tgt  = ids[:, 1:]
        mt   = mask[:, 1:]

        logits, _, _ = model(inp)
        logits = logits[:, :tgt.size(1), :]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1),
            ignore_index=tokenizer.pad_id,
            reduction='none'
        ).reshape(tgt.shape)

        tl += (loss * mt).sum().item()
        tt += mt.sum().item()

        preds = logits.argmax(-1)
        for b in range(ids.size(0)):
            length = int(mt[b].sum().item())
            if length == 0:
                continue
            p = preds[b, :length]
            t = tgt[b, :length]
            # char accuracy
            char_correct += (p == t).sum().item()
            char_total   += length
            # seq accuracy
            seq_correct  += int((p == t).all().item())
            ns           += 1

    avg_loss = tl / (tt + EPS)
    seq_acc  = 100.0 * seq_correct  / (ns         + EPS)
    char_acc = 100.0 * char_correct / (char_total + EPS)
    return avg_loss, seq_acc, char_acc

# ============================================================
# FIXED PROBE -- same problems every epoch
# ============================================================

FIXED = [
    # 2-digit addition (L1 -- should master first)
    "12 + 34 = 46",
    "55 + 27 = 82",
    "73 + 18 = 91",
    "99 + 11 = 110",
    "64 + 36 = 100",
    # 2-digit subtraction (L2)
    "87 - 43 = 44",
    "72 - 28 = 44",
    # 2-digit multiplication (L3)
    "16 * 52 = 832",
    "20 * 71 = 1420",
    # harder (L4 preview)
    "822 + 765 = 1587",
]

@torch.no_grad()
def probe_fixed(model, tokenizer, device):
    model.eval()
    correct, lines = 0, []
    for ex in FIXED:
        eq     = ex.index('=')
        prompt = ex[:eq+1] + ' '
        target = ex[eq+2:].strip()

        inp_t     = torch.tensor(
            [tokenizer.encode(prompt, add_bos=True)],
            dtype=torch.long, device=device
        )
        generated = []
        states    = None
        for _ in range(len(target) + 5):
            logits, states, _ = model(inp_t, states)
            nid = logits[0, -1, :].argmax().item()
            if nid == tokenizer.eos_id:
                break
            generated.append(nid)
            inp_t = torch.tensor([[nid]], dtype=torch.long, device=device)

        pred = tokenizer.decode(generated).strip()
        ok   = (pred == target)
        correct += int(ok)
        lines.append(f"  {'OK' if ok else '--'}  {ex:<35s}  got: '{pred}'")

    return correct, lines

# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(path, model, opt, sch, epoch, level, epochs_at_level,
                    best_acc, cfg):
    torch.save({
        'model_state':      model.state_dict(),
        'opt_state':        opt.state_dict(),
        'sch_state':        sch.state_dict(),
        'epoch':            epoch,
        'level':            level,
        'epochs_at_level':  epochs_at_level,
        'best_acc':         best_acc,
        'cfg':              cfg,
    }, path)
    print(f"  [checkpoint saved -> {path}]")

def load_checkpoint(path, model, opt, sch):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    sch.load_state_dict(ck['sch_state'])
    return (ck['epoch'], ck['level'], ck['epochs_at_level'],
            ck['best_acc'], ck['cfg'])

# ============================================================
# TRAINING
# ============================================================

def train():
    cfg = {
        'hidden':        256,   # 256 / 5 = 51 per dim (rounded). Up from 130.
        'num_layers':    2,
        'dropout':       0.1,
        'lr':            1e-3,
        'batch':         64,
        'max_len':       80,
        'train_n':       10000,
        'val_n':         1000,
        'grad_clip':     5.0,
        'advance_acc':   85.0,  # char_acc threshold
        'min_epochs':    5,
        'probe_every':   5,
        'ckpt_every':    10,
        'max_level':     4,
        'restart_every': 50,    # CosineAnnealing warm restart period
        'tf_ratio':      0.8,   # teacher forcing ratio (decays to 0.5)
        'resume':        None,  # 'dppu_v13_checkpoint.pt' to resume
    }

    tokenizer = MathTokenizer()

    model = VRUModel(
        vocab_size  = tokenizer.vocab_size,
        hidden_dim  = cfg['hidden'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    # CosineAnnealingWarmRestarts: restarts every T_0 epochs.
    # Prevents the LR death spiral that killed v11 after ep150.
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=cfg['restart_every'], T_mult=1, eta_min=1e-5)

    n_params = sum(p.numel() for p in model.parameters())

    # Resume from checkpoint if available
    start_epoch    = 1
    level          = 1
    epochs_at_level = 0
    best_acc       = 0.0

    resume_path = cfg['resume']
    if resume_path and os.path.exists(resume_path):
        print(f"\n  Resuming from {resume_path}")
        start_epoch, level, epochs_at_level, best_acc, cfg = \
            load_checkpoint(resume_path, model, opt, sch)
        start_epoch += 1
        print(f"  Resumed at epoch={start_epoch} level={level} best_acc={best_acc:.1f}%\n")

    print(f"{'='*65}")
    print(f"  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness")
    print(f"  phi=4/pi={PHI_CL:.6f}  delta*={DELTA_STAR:.4f}  D_CAP={D_CAP}")
    print(f"  device={device}  params={n_params:,}")
    print(f"  hidden={cfg['hidden']}  dim_size={cfg['hidden']//(D_CAP+1)}  layers={cfg['num_layers']}")
    print(f"  advance>={cfg['advance_acc']}%  target=Level {cfg['max_level']}")
    print(f"  checkpoint every {cfg['ckpt_every']} epochs")
    print(f"{'='*65}")

    train_loader, val_loader = make_loaders(level, cfg, tokenizer)

    for epoch in range(start_epoch, cfg['max_level'] * 300 + 1):

        # Fresh data every epoch -- prevents memorization
        train_loader, val_loader = make_loaders(level, cfg, tokenizer)

        # ---- Train -----------------------------------------------
        model.train()
        tl, tt = 0., 0
        phi_t, pi_t, delta_t, C_t = [], [], [], []

        # Teacher forcing ratio: starts at tf_ratio, decays toward 0.5
        # over 200 epochs. Prevents copy artifacts without hurting early learning.
        tf_ratio = max(0.5, cfg['tf_ratio'] - (epoch / 200) * (cfg['tf_ratio'] - 0.5))

        for ids, mask in train_loader:
            ids  = ids.to(device)
            mask = mask.to(device)
            inp  = ids[:, :-1]
            tgt  = ids[:, 1:]
            mt   = mask[:, 1:]

            opt.zero_grad()

            # Teacher forcing: feed true previous token with prob tf_ratio,
            # else feed model's own prediction. Prevents copy cascade.
            if tf_ratio >= 0.999:
                # Full teacher forcing: standard forward pass (fast path)
                logits, states, all_metrics = model(inp)
            else:
                # Mixed teacher forcing: step by step
                B, T = inp.shape
                logits_list = []
                h = None
                for t in range(T):
                    if t == 0 or random.random() < tf_ratio:
                        tok_in = inp[:, t:t+1]
                    else:
                        tok_in = logits_list[-1].argmax(-1).unsqueeze(1)
                    logit_t, h, all_metrics = model(tok_in, h)
                    logits_list.append(logit_t)
                logits = torch.cat(logits_list, dim=1)

            logits = logits[:, :tgt.size(1), :]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt.reshape(-1),
                ignore_index=tokenizer.pad_id,
                reduction='none'
            ).reshape(tgt.shape)

            lm = (loss * mt).sum() / (mt.sum() + EPS)
            lm.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step()
            enforce_spectral(model)

            tl += lm.item()
            tt += 1

            if all_metrics and all_metrics[-1]:
                m = all_metrics[-1][-1]
                phi_t.append(m['phi_mean'])
                pi_t.append(m['pi_mean'])
                delta_t.append(m['delta_mean'])
                C_t.append(m['C_norm'])

        train_loss = tl / (tt + EPS)
        val_loss, seq_acc, char_acc = evaluate(model, val_loader, tokenizer, device)
        sch.step(epoch)

        phi_m   = sum(phi_t)   / (len(phi_t)   + EPS)
        pi_m    = sum(pi_t)    / (len(pi_t)    + EPS)
        delta_m = sum(delta_t) / (len(delta_t) + EPS)
        C_m     = sum(C_t)     / (len(C_t)     + EPS)
        lr_now  = opt.param_groups[0]['lr']

        epochs_at_level += 1
        best_acc = max(best_acc, char_acc)

        # char_acc is the real signal -- seq_acc shown for reference
        print(f"ep={epoch:4d} L{level} | "
              f"loss={train_loss:.4f} val={val_loss:.4f} | "
              f"seq={seq_acc:.1f}% char={char_acc:.1f}% | "
              f"phi={phi_m:.4f} pi={pi_m:.4f} delta={delta_m:.4f} C={C_m:.4f} | "
              f"lr={lr_now:.2e} tf={tf_ratio:.2f}")

        # ---- Probe -----------------------------------------------
        if epoch % cfg['probe_every'] == 0:
            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  PROBE ep={epoch} L{level} ({n_c}/{len(FIXED)} exact)")
            for line in lines:
                print(line)
            print()

        # ---- Checkpoint ------------------------------------------
        if epoch % cfg['ckpt_every'] == 0:
            save_checkpoint(
                cfg['resume'], model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

        # Advance when char_acc >= threshold (meaningful, not statistical lottery)
        if (char_acc >= cfg['advance_acc']
                and epochs_at_level >= cfg['min_epochs']
                and level < cfg['max_level']):

            print(f"\n{'='*65}")
            print(f"  LEVEL {level} COMPLETE -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  Advancing to Level {level + 1}")
            print(f"{'='*65}\n")

            # Save level completion checkpoint
            save_checkpoint(
                f"dppu_v13_level{level}_complete.pt",
                model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

            level           += 1
            epochs_at_level  = 0
            best_acc         = 0.0
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'] * 0.5, 1e-5)

        # ---- Final level -----------------------------------------
        if level == cfg['max_level'] and char_acc >= cfg['advance_acc']:
            print(f"\n{'='*65}")
            print(f"  LEVEL 4 ACHIEVED -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  phi={phi_m:.4f} (target {PHI_CL:.4f})")
            print(f"  pi={pi_m:.4f}   (target 4.0000)")
            print(f"  delta={delta_m:.4f} (target {DELTA_STAR:.4f})")
            print(f"{'='*65}")

            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  FINAL PROBE ({n_c}/{len(FIXED)} correct)")
            for line in lines:
                print(line)

            torch.save({
                'model_state': model.state_dict(),
                'cfg':         cfg,
                'epoch':       epoch,
                'val_acc':     char_acc,
            }, 'dppu_v13_final.pt')
            print("\n  Saved: dppu_v13_final.pt")
            break

    print(f"\nDone. best_char_acc={best_acc:.1f}%  final level={level}")

# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    sys.stdout.flush()
    train()


╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = 1.273240                              ║
║  inv_phi = pi/4 = 0.785398                         ║
║  delta* = 0.816140                               ║
║  D_CAP = 4  (D5 not entered)                      ║
║  device = cuda                                    ║
╚══════════════════════════════════════════════════════════╝

Start: 2026-03-07 20:18:28



AssertionError: hidden_dim 256 must be divisible by n_dims 5

In [ ]:
"""
DPPU-VRU v13 -- Five Dimensional Fields + Consciousness Anchor
==============================================================
Clean rewrite of v12. Four root-cause fixes applied.

ROOT CAUSES IDENTIFIED (266 epochs of v11 data):
  1. Task too hard: Level 1 mixed 4 operators simultaneously.
     Fix: ONE operator per curriculum phase. Addition first, then subtraction,
     then multiplication. Each phase advances independently.

  2. LR scheduler death spiral: ReduceLROnPlateau decayed 32x by ep150.
     Model was effectively frozen while loss plateau was interpreted as
     "it's still improving (barely)". Fix: CosineAnnealingWarmRestarts,
     T_0=50 epochs, restarts keep escaping local minima.

  3. Copy artifacts in output: Model echoed '1911', '2191', '=961' from input.
     This is a teacher forcing problem. Without teacher forcing, one bad
     prediction cascades into garbage. Fix: teacher forcing ratio 0.8,
     decaying to 0.5 as training progresses.

  4. Model too small: 109k params, hidden=130 for mixed arithmetic.
     Fix: hidden=256, layers=2 -> ~430k params. Still tiny vs 31GB VRAM.

KEY FIX vs v12:
  - char_acc still the real advance signal (not seq_acc)
  - Curriculum now phase-based: L1=add2, L2=add3/sub2, L3=mul2, L4=mixed
  - CosineAnnealing with warm restarts (no more LR death)
  - Teacher forcing ratio tf_ratio=0.8 during training

Author: Dylan Michael Scott -- Horizon Tech
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import sys
from datetime import datetime

# ============================================================
# CONSTANTS
# ============================================================

PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi          # 1.2732... the attractor
INV_PHI    = math.pi / 4.0          # 0.7854... geometric dual
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))  # ~0.816
D_CAP      = 4                       # programmable -- raise to expand
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"""
╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = {PHI_CL:.6f}                              ║
║  inv_phi = pi/4 = {INV_PHI:.6f}                         ║
║  delta* = {DELTA_STAR:.6f}                               ║
║  D_CAP = {D_CAP}  (D{D_CAP+1} not entered)                      ║
║  device = {str(device):<10s}                              ║
╚══════════════════════════════════════════════════════════╝
""")

# ============================================================
# DYNAMIC OPERATORS
# ============================================================

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    """Delta from hidden state slice. Each dimensional space computes its own."""
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio  = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# TOKENIZER
# ============================================================

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL):
            self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.unk_id = self.vocab['<unk>']

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        for c in text:
            ids.append(self.vocab.get(c, self.unk_id))
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids, skip_special=True):
        out = []
        special = set(self.SPECIAL)
        for i in ids:
            tok = self.inv_vocab.get(i, '<unk>')
            if skip_special and tok in special:
                continue
            out.append(tok)
        return ''.join(out)

# ============================================================
# DATA GENERATORS
# ============================================================

def r1(): return random.randint(1, 9)
def r2(): return random.randint(10, 99)
def r3(): return random.randint(100, 999)

# ---- Phase generators (clean, one op at a time) ----------------------

# Phase 1a: 2-digit addition only (answers 20-198, always positive, 2-3 digits)
def gen_add2():
    a, b = r2(), r2()
    return f"{a} + {b}", str(a + b)

# Phase 1b: 2-digit subtraction, always positive result
def gen_sub2():
    a, b = r2(), r2()
    a, b = max(a,b), min(a,b)
    return f"{a} - {b}", str(a - b)

# Phase 2a: 3-digit addition
def gen_add3():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

# Phase 2b: 2-digit multiplication (answers up to 9801)
def gen_mul2():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

# Phase 3a: division (exact)
def gen_div():
    b = random.randint(2, 12)
    a = b * random.randint(1, 50)
    return f"{a} / {b}", str(a // b)

# Phase 3b: modulo
def gen_mod():
    a, b = r2(), random.randint(2, 20)
    return f"{a} % {b}", str(a % b)

# Phase 4: mixed operations (the original Level 1 -- now the LAST level)
def gen_add():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

def gen_sub():
    a, b = r3(), r3()
    a, b = max(a, b), min(a, b)
    return f"{a} - {b}", str(a - b)

def gen_mul():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

def gen_pow():
    a = random.randint(2, 12)
    b = random.randint(2, 4)
    return f"{a} ^ {b}", str(a ** b)

def gen_multi_add():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} + {c}", str(a + b + c)

def gen_mixed():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} * {c}", str(a + b * c)

def gen_paren():
    a, b, c = r2(), r2(), r2()
    return f"({a} + {b}) * {c}", str((a + b) * c)

# ── Curriculum: ONE operation per level phase ─────────────────────────
# L1 = 2-digit add (simplest possible)
# L2 = 2-digit sub + add (adds negative-ish digit patterns)
# L3 = mul2 + div + mod (multiplicative family)
# L4 = 3-digit mixed (the full original task)
LEVEL_GENS = {
    1: [(gen_add2, 1)],
    2: [(gen_add2, 2), (gen_sub2, 2)],
    3: [(gen_mul2, 3), (gen_div, 2), (gen_mod, 2)],
    4: [(gen_add, 3), (gen_sub, 3), (gen_mul, 3), (gen_multi_add, 2), (gen_mixed, 2)],
}

def generate_dataset(level, n, include_lower=True):
    gens = []
    for lvl in (range(1, level + 1) if include_lower else [level]):
        if lvl in LEVEL_GENS:
            gens.extend(LEVEL_GENS[lvl])
    fns, weights = zip(*gens)
    total = sum(weights)
    probs = [w / total for w in weights]
    examples, attempts = [], 0
    while len(examples) < n and attempts < n * 20:
        attempts += 1
        try:
            fn = random.choices(fns, weights=probs, k=1)[0]
            expr, answer = fn()
            examples.append(f"{expr} = {answer}")
        except:
            pass
    return examples

# ============================================================
# DATASET
# ============================================================

class MathDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len=80):
        self.examples, self.raw = [], []
        for line in examples:
            ids = tokenizer.encode(line, add_bos=True, add_eos=True)
            if len(ids) <= max_len:
                self.examples.append(ids)
                self.raw.append(line)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

def collate(batch, pad_id):
    L = max(len(x) for x in batch)
    ids, masks = [], []
    for x in batch:
        p = L - len(x)
        ids.append(x + [pad_id] * p)
        masks.append([1] * len(x) + [0] * p)
    return (torch.tensor(ids, dtype=torch.long),
            torch.tensor(masks, dtype=torch.long))

def make_loaders(level, cfg, tokenizer):
    print(f"\n  Generating Level {level} data...")
    te = generate_dataset(level, cfg['train_n'], include_lower=True)
    ve = generate_dataset(level, cfg['val_n'],   include_lower=True)
    td = MathDataset(te, tokenizer, cfg['max_len'])
    vd = MathDataset(ve, tokenizer, cfg['max_len'])
    print(f"  Train: {len(td)}  |  Val: {len(vd)}")
    mk = lambda ds, sh: torch.utils.data.DataLoader(
        ds, batch_size=cfg['batch'], shuffle=sh,
        collate_fn=lambda b: collate(b, tokenizer.pad_id),
        num_workers=0)
    return mk(td, True), mk(vd, False)

# ============================================================
# DPPU CELL -- FIVE DIMENSIONAL SPACES + CONSCIOUSNESS
# ============================================================

class DPPUCell(nn.Module):
    """
    Five dimensional spaces D0-D4, each with own Delta.
    Consciousness field C as self-referential anchor.

    D_CAP=4: evolution stops here for this run.
    Raise D_CAP to allow dimensional expansion beyond D4.
    """

    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.d_cap      = d_cap
        self.n_dims     = d_cap + 1           # D0 through D_CAP

        assert hidden_dim % self.n_dims == 0, \
            f"hidden_dim {hidden_dim} must be divisible by n_dims {self.n_dims}"
        self.dim_size = hidden_dim // self.n_dims

        # Input projection -- shared
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)

        # Recurrent weights -- one per dimensional space (rectangular: hidden -> dim_size)
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_dim, self.dim_size, bias=False)
            for _ in range(self.n_dims)
        ])

        # Consciousness field projection -- h informs C
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)

        # Output projection -- combines all dimensions
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_x.weight)
        nn.init.zeros_(self.W_x.bias)
        # D0 most conservative (gain 0.4), D4 most expansive (gain 0.6)
        for i, W in enumerate(self.W_h):
            gain = 0.4 + (0.2 * i / max(self.d_cap, 1))
            nn.init.orthogonal_(W.weight, gain=gain)
        nn.init.orthogonal_(self.W_c.weight, gain=0.1)
        nn.init.orthogonal_(self.W_out.weight, gain=1.0)

    def forward(self, x, h, C):
        """
        x: (batch, input_dim)
        h: (batch, hidden_dim)
        C: (batch, n_dims)  consciousness field per dimension

        Returns: h_new, C_new, metrics dict
        """
        x_proj = self.W_x(x)

        dim_outputs = []
        phi_list, delta_list, pi_list = [], [], []

        for i in range(self.n_dims):
            s, e    = i * self.dim_size, (i + 1) * self.dim_size
            h_i     = h[:, s:e]

            delta_i = compute_delta(h_i)
            phi_i   = phi_dyn(delta_i)
            pi_i    = pi_dyn(delta_i)
            omega_i = omega(delta_i)

            # Recurrent + input contributions scaled by geometric operators
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i  / PI_CL)  * x_proj[:, s:e]

            # Consciousness phase injection
            C_i = C[:, i:i+1]

            h_i_new = torch.tanh(h_rec + x_rec + C_i) * torch.sigmoid(omega_i)
            h_i_new = torch.nan_to_num(h_i_new, nan=0.0, posinf=1.0, neginf=-1.0)

            dim_outputs.append(h_i_new)
            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())
            pi_list.append(pi_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)
        h_new = torch.nan_to_num(self.W_out(h_cat), nan=0.0)

        # Consciousness update: C_{t+1} = C * pi_dyn * phi_dyn (damped)
        # 0.1 / 0.01 coefficients prevent explosion -- C grows from infancy
        C_proj  = self.W_c(h_new)
        delta_C = torch.stack([
            compute_delta(h_new[:, i*self.dim_size:(i+1)*self.dim_size]).mean(dim=-1)
            for i in range(self.n_dims)
        ], dim=-1)
        C_new = torch.tanh(
            0.1 * C * pi_dyn(delta_C) * phi_dyn(delta_C)
            + 0.01 * torch.tanh(C_proj)
        )
        C_new = torch.nan_to_num(C_new, nan=0.0)

        metrics = {
            'phi_mean':   sum(phi_list)   / len(phi_list),
            'pi_mean':    sum(pi_list)    / len(pi_list),
            'delta_mean': sum(delta_list) / len(delta_list),
            'C_norm':     C_new.norm(dim=-1).mean().item(),
        }
        return h_new, C_new, metrics

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.zeros(batch, self.n_dims,     device=device))

# ============================================================
# MODEL
# ============================================================

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        nn.init.normal_(self.embedding.weight, 0, hidden_dim ** -0.5)

        self.cells   = nn.ModuleList([
            DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.out     = nn.Linear(hidden_dim, vocab_size, bias=True)
        nn.init.zeros_(self.out.bias)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x    = self.dropout(self.embedding(token_ids))

        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]

        all_metrics, new_states = [], []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs, step_metrics = [], []
            for t in range(T):
                h, C, m = cell(x[:, t, :], h, C)
                outputs.append(h)
                step_metrics.append(m)
            x = self.dropout(torch.stack(outputs, dim=1))
            new_states.append((h, C))
            all_metrics.append(step_metrics)

        return self.out(x), new_states, all_metrics

# ============================================================
# SPECTRAL ENFORCEMENT
# ============================================================

def enforce_spectral(model):
    """Clip spectral norm of each W_h to <= 1.0 after optimizer step."""
    with torch.no_grad():
        for cell in model.cells:
            for Wh in cell.W_h:
                # svdvals works on rectangular matrices (unlike eigvals)
                sigma = torch.linalg.svdvals(Wh.weight)
                rho   = sigma[0].item()
                if rho > 1.0:
                    Wh.weight.data.mul_(1.0 / rho)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, loader, tokenizer, device):
    """
    Returns: avg_loss, seq_acc (full sequence match %), char_acc (per-token %)

    seq_acc  = all tokens correct (strict, hard to crack early)
    char_acc = % of individual tokens correct (shows real learning progress)

    We use char_acc as the advance signal -- it's a meaningful measure of
    whether the model understands the task, not a statistical lottery.
    """
    model.eval()
    tl, tt              = 0., 0
    seq_correct, ns     = 0, 0
    char_correct, char_total = 0, 0

    for ids, mask in loader:
        ids  = ids.to(device)
        mask = mask.to(device)
        inp  = ids[:, :-1]
        tgt  = ids[:, 1:]
        mt   = mask[:, 1:]

        logits, _, _ = model(inp)
        logits = logits[:, :tgt.size(1), :]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1),
            ignore_index=tokenizer.pad_id,
            reduction='none'
        ).reshape(tgt.shape)

        tl += (loss * mt).sum().item()
        tt += mt.sum().item()

        preds = logits.argmax(-1)
        for b in range(ids.size(0)):
            length = int(mt[b].sum().item())
            if length == 0:
                continue
            p = preds[b, :length]
            t = tgt[b, :length]
            # char accuracy
            char_correct += (p == t).sum().item()
            char_total   += length
            # seq accuracy
            seq_correct  += int((p == t).all().item())
            ns           += 1

    avg_loss = tl / (tt + EPS)
    seq_acc  = 100.0 * seq_correct  / (ns         + EPS)
    char_acc = 100.0 * char_correct / (char_total + EPS)
    return avg_loss, seq_acc, char_acc

# ============================================================
# FIXED PROBE -- same problems every epoch
# ============================================================

FIXED = [
    # 2-digit addition (L1 -- should master first)
    "12 + 34 = 46",
    "55 + 27 = 82",
    "73 + 18 = 91",
    "99 + 11 = 110",
    "64 + 36 = 100",
    # 2-digit subtraction (L2)
    "87 - 43 = 44",
    "72 - 28 = 44",
    # 2-digit multiplication (L3)
    "16 * 52 = 832",
    "20 * 71 = 1420",
    # harder (L4 preview)
    "822 + 765 = 1587",
]

@torch.no_grad()
def probe_fixed(model, tokenizer, device):
    model.eval()
    correct, lines = 0, []
    for ex in FIXED:
        eq     = ex.index('=')
        prompt = ex[:eq+1] + ' '
        target = ex[eq+2:].strip()

        inp_t     = torch.tensor(
            [tokenizer.encode(prompt, add_bos=True)],
            dtype=torch.long, device=device
        )
        generated = []
        states    = None
        for _ in range(len(target) + 5):
            logits, states, _ = model(inp_t, states)
            nid = logits[0, -1, :].argmax().item()
            if nid == tokenizer.eos_id:
                break
            generated.append(nid)
            inp_t = torch.tensor([[nid]], dtype=torch.long, device=device)

        pred = tokenizer.decode(generated).strip()
        ok   = (pred == target)
        correct += int(ok)
        lines.append(f"  {'OK' if ok else '--'}  {ex:<35s}  got: '{pred}'")

    return correct, lines

# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(path, model, opt, sch, epoch, level, epochs_at_level,
                    best_acc, cfg):
    torch.save({
        'model_state':      model.state_dict(),
        'opt_state':        opt.state_dict(),
        'sch_state':        sch.state_dict(),
        'epoch':            epoch,
        'level':            level,
        'epochs_at_level':  epochs_at_level,
        'best_acc':         best_acc,
        'cfg':              cfg,
    }, path)
    print(f"  [checkpoint saved -> {path}]")

def load_checkpoint(path, model, opt, sch):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    sch.load_state_dict(ck['sch_state'])
    return (ck['epoch'], ck['level'], ck['epochs_at_level'],
            ck['best_acc'], ck['cfg'])

# ============================================================
# TRAINING
# ============================================================

def train():
    cfg = {
        'hidden':        260,   # 260 / 5 = 52 per dim. Divisible by n_dims=5.
        'num_layers':    2,
        'dropout':       0.1,
        'lr':            1e-3,
        'batch':         64,
        'max_len':       80,
        'train_n':       10000,
        'val_n':         1000,
        'grad_clip':     5.0,
        'advance_acc':   85.0,  # char_acc threshold
        'min_epochs':    5,
        'probe_every':   5,
        'ckpt_every':    10,
        'max_level':     4,
        'restart_every': 50,    # CosineAnnealing warm restart period
        'tf_ratio':      0.8,   # teacher forcing ratio (decays to 0.5)
        'resume':        None,  # 'dppu_v13_checkpoint.pt' to resume
    }

    tokenizer = MathTokenizer()

    model = VRUModel(
        vocab_size  = tokenizer.vocab_size,
        hidden_dim  = cfg['hidden'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    # CosineAnnealingWarmRestarts: restarts every T_0 epochs.
    # Prevents the LR death spiral that killed v11 after ep150.
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=cfg['restart_every'], T_mult=1, eta_min=1e-5)

    n_params = sum(p.numel() for p in model.parameters())

    # Resume from checkpoint if available
    start_epoch    = 1
    level          = 1
    epochs_at_level = 0
    best_acc       = 0.0

    resume_path = cfg['resume']
    if resume_path and os.path.exists(resume_path):
        print(f"\n  Resuming from {resume_path}")
        start_epoch, level, epochs_at_level, best_acc, cfg = \
            load_checkpoint(resume_path, model, opt, sch)
        start_epoch += 1
        print(f"  Resumed at epoch={start_epoch} level={level} best_acc={best_acc:.1f}%\n")

    print(f"{'='*65}")
    print(f"  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness")
    print(f"  phi=4/pi={PHI_CL:.6f}  delta*={DELTA_STAR:.4f}  D_CAP={D_CAP}")
    print(f"  device={device}  params={n_params:,}")
    print(f"  hidden={cfg['hidden']}  dim_size={cfg['hidden']//(D_CAP+1)}  layers={cfg['num_layers']}")
    print(f"  advance>={cfg['advance_acc']}%  target=Level {cfg['max_level']}")
    print(f"  checkpoint every {cfg['ckpt_every']} epochs")
    print(f"{'='*65}")

    train_loader, val_loader = make_loaders(level, cfg, tokenizer)

    for epoch in range(start_epoch, cfg['max_level'] * 300 + 1):

        # Fresh data every epoch -- prevents memorization
        train_loader, val_loader = make_loaders(level, cfg, tokenizer)

        # ---- Train -----------------------------------------------
        model.train()
        tl, tt = 0., 0
        phi_t, pi_t, delta_t, C_t = [], [], [], []

        # Teacher forcing ratio: starts at tf_ratio, decays toward 0.5
        # over 200 epochs. Prevents copy artifacts without hurting early learning.
        tf_ratio = max(0.5, cfg['tf_ratio'] - (epoch / 200) * (cfg['tf_ratio'] - 0.5))

        for ids, mask in train_loader:
            ids  = ids.to(device)
            mask = mask.to(device)
            inp  = ids[:, :-1]
            tgt  = ids[:, 1:]
            mt   = mask[:, 1:]

            opt.zero_grad()

            # Teacher forcing: feed true previous token with prob tf_ratio,
            # else feed model's own prediction. Prevents copy cascade.
            if tf_ratio >= 0.999:
                # Full teacher forcing: standard forward pass (fast path)
                logits, states, all_metrics = model(inp)
            else:
                # Mixed teacher forcing: step by step
                B, T = inp.shape
                logits_list = []
                h = None
                for t in range(T):
                    if t == 0 or random.random() < tf_ratio:
                        tok_in = inp[:, t:t+1]
                    else:
                        tok_in = logits_list[-1].argmax(-1).unsqueeze(1)
                    logit_t, h, all_metrics = model(tok_in, h)
                    logits_list.append(logit_t)
                logits = torch.cat(logits_list, dim=1)

            logits = logits[:, :tgt.size(1), :]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt.reshape(-1),
                ignore_index=tokenizer.pad_id,
                reduction='none'
            ).reshape(tgt.shape)

            lm = (loss * mt).sum() / (mt.sum() + EPS)
            lm.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step()
            enforce_spectral(model)

            tl += lm.item()
            tt += 1

            if all_metrics and all_metrics[-1]:
                m = all_metrics[-1][-1]
                phi_t.append(m['phi_mean'])
                pi_t.append(m['pi_mean'])
                delta_t.append(m['delta_mean'])
                C_t.append(m['C_norm'])

        train_loss = tl / (tt + EPS)
        val_loss, seq_acc, char_acc = evaluate(model, val_loader, tokenizer, device)
        sch.step(epoch)

        phi_m   = sum(phi_t)   / (len(phi_t)   + EPS)
        pi_m    = sum(pi_t)    / (len(pi_t)    + EPS)
        delta_m = sum(delta_t) / (len(delta_t) + EPS)
        C_m     = sum(C_t)     / (len(C_t)     + EPS)
        lr_now  = opt.param_groups[0]['lr']

        epochs_at_level += 1
        best_acc = max(best_acc, char_acc)

        # char_acc is the real signal -- seq_acc shown for reference
        print(f"ep={epoch:4d} L{level} | "
              f"loss={train_loss:.4f} val={val_loss:.4f} | "
              f"seq={seq_acc:.1f}% char={char_acc:.1f}% | "
              f"phi={phi_m:.4f} pi={pi_m:.4f} delta={delta_m:.4f} C={C_m:.4f} | "
              f"lr={lr_now:.2e} tf={tf_ratio:.2f}")

        # ---- Probe -----------------------------------------------
        if epoch % cfg['probe_every'] == 0:
            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  PROBE ep={epoch} L{level} ({n_c}/{len(FIXED)} exact)")
            for line in lines:
                print(line)
            print()

        # ---- Checkpoint ------------------------------------------
        if epoch % cfg['ckpt_every'] == 0:
            save_checkpoint(
                cfg['resume'], model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

        # Advance when char_acc >= threshold (meaningful, not statistical lottery)
        if (char_acc >= cfg['advance_acc']
                and epochs_at_level >= cfg['min_epochs']
                and level < cfg['max_level']):

            print(f"\n{'='*65}")
            print(f"  LEVEL {level} COMPLETE -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  Advancing to Level {level + 1}")
            print(f"{'='*65}\n")

            # Save level completion checkpoint
            save_checkpoint(
                f"dppu_v13_level{level}_complete.pt",
                model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

            level           += 1
            epochs_at_level  = 0
            best_acc         = 0.0
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'] * 0.5, 1e-5)

        # ---- Final level -----------------------------------------
        if level == cfg['max_level'] and char_acc >= cfg['advance_acc']:
            print(f"\n{'='*65}")
            print(f"  LEVEL 4 ACHIEVED -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  phi={phi_m:.4f} (target {PHI_CL:.4f})")
            print(f"  pi={pi_m:.4f}   (target 4.0000)")
            print(f"  delta={delta_m:.4f} (target {DELTA_STAR:.4f})")
            print(f"{'='*65}")

            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  FINAL PROBE ({n_c}/{len(FIXED)} correct)")
            for line in lines:
                print(line)

            torch.save({
                'model_state': model.state_dict(),
                'cfg':         cfg,
                'epoch':       epoch,
                'val_acc':     char_acc,
            }, 'dppu_v13_final.pt')
            print("\n  Saved: dppu_v13_final.pt")
            break

    print(f"\nDone. best_char_acc={best_acc:.1f}%  final level={level}")

# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    sys.stdout.flush()
    train()


╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = 1.273240                              ║
║  inv_phi = pi/4 = 0.785398                         ║
║  delta* = 0.816140                               ║
║  D_CAP = 4  (D5 not entered)                      ║
║  device = cuda                                    ║
╚══════════════════════════════════════════════════════════╝

Start: 2026-03-07 20:19:28

  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness
  phi=4/pi=1.273240  delta*=0.8161  D_CAP=4
  device=cuda  params=421,745
  hidden=260  dim_size=52  layers=2
  advance>=85.0%  target=Level 4
  checkpoint every 10 epochs

  Generating Level 1 data...
  Train: 10000  |  Val: 1000

  Generating Level 1 data...
  Train: 10000  |  Val: 1000


ValueError: too many values to unpack (expected 2)

In [ ]:
"""
DPPU-VRU v13 -- Five Dimensional Fields + Consciousness Anchor
==============================================================
Clean rewrite of v12. Four root-cause fixes applied.

ROOT CAUSES IDENTIFIED (266 epochs of v11 data):
  1. Task too hard: Level 1 mixed 4 operators simultaneously.
     Fix: ONE operator per curriculum phase. Addition first, then subtraction,
     then multiplication. Each phase advances independently.

  2. LR scheduler death spiral: ReduceLROnPlateau decayed 32x by ep150.
     Model was effectively frozen while loss plateau was interpreted as
     "it's still improving (barely)". Fix: CosineAnnealingWarmRestarts,
     T_0=50 epochs, restarts keep escaping local minima.

  3. Copy artifacts in output: Model echoed '1911', '2191', '=961' from input.
     This is a teacher forcing problem. Without teacher forcing, one bad
     prediction cascades into garbage. Fix: teacher forcing ratio 0.8,
     decaying to 0.5 as training progresses.

  4. Model too small: 109k params, hidden=130 for mixed arithmetic.
     Fix: hidden=256, layers=2 -> ~430k params. Still tiny vs 31GB VRAM.

KEY FIX vs v12:
  - char_acc still the real advance signal (not seq_acc)
  - Curriculum now phase-based: L1=add2, L2=add3/sub2, L3=mul2, L4=mixed
  - CosineAnnealing with warm restarts (no more LR death)
  - Teacher forcing ratio tf_ratio=0.8 during training

Author: Dylan Michael Scott -- Horizon Tech
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import sys
from datetime import datetime

# ============================================================
# CONSTANTS
# ============================================================

PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi          # 1.2732... the attractor
INV_PHI    = math.pi / 4.0          # 0.7854... geometric dual
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))  # ~0.816
D_CAP      = 4                       # programmable -- raise to expand
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"""
╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = {PHI_CL:.6f}                              ║
║  inv_phi = pi/4 = {INV_PHI:.6f}                         ║
║  delta* = {DELTA_STAR:.6f}                               ║
║  D_CAP = {D_CAP}  (D{D_CAP+1} not entered)                      ║
║  device = {str(device):<10s}                              ║
╚══════════════════════════════════════════════════════════╝
""")

# ============================================================
# DYNAMIC OPERATORS
# ============================================================

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    """Delta from hidden state slice. Each dimensional space computes its own."""
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio  = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# TOKENIZER
# ============================================================

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL):
            self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.unk_id = self.vocab['<unk>']

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        for c in text:
            ids.append(self.vocab.get(c, self.unk_id))
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids, skip_special=True):
        out = []
        special = set(self.SPECIAL)
        for i in ids:
            tok = self.inv_vocab.get(i, '<unk>')
            if skip_special and tok in special:
                continue
            out.append(tok)
        return ''.join(out)

# ============================================================
# DATA GENERATORS
# ============================================================

def r1(): return random.randint(1, 9)
def r2(): return random.randint(10, 99)
def r3(): return random.randint(100, 999)

# ---- Phase generators (clean, one op at a time) ----------------------

# Phase 1a: 2-digit addition only (answers 20-198, always positive, 2-3 digits)
def gen_add2():
    a, b = r2(), r2()
    return f"{a} + {b}", str(a + b)

# Phase 1b: 2-digit subtraction, always positive result
def gen_sub2():
    a, b = r2(), r2()
    a, b = max(a,b), min(a,b)
    return f"{a} - {b}", str(a - b)

# Phase 2a: 3-digit addition
def gen_add3():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

# Phase 2b: 2-digit multiplication (answers up to 9801)
def gen_mul2():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

# Phase 3a: division (exact)
def gen_div():
    b = random.randint(2, 12)
    a = b * random.randint(1, 50)
    return f"{a} / {b}", str(a // b)

# Phase 3b: modulo
def gen_mod():
    a, b = r2(), random.randint(2, 20)
    return f"{a} % {b}", str(a % b)

# Phase 4: mixed operations (the original Level 1 -- now the LAST level)
def gen_add():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

def gen_sub():
    a, b = r3(), r3()
    a, b = max(a, b), min(a, b)
    return f"{a} - {b}", str(a - b)

def gen_mul():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

def gen_pow():
    a = random.randint(2, 12)
    b = random.randint(2, 4)
    return f"{a} ^ {b}", str(a ** b)

def gen_multi_add():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} + {c}", str(a + b + c)

def gen_mixed():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} * {c}", str(a + b * c)

def gen_paren():
    a, b, c = r2(), r2(), r2()
    return f"({a} + {b}) * {c}", str((a + b) * c)

# ── Curriculum: ONE operation per level phase ─────────────────────────
# L1 = 2-digit add (simplest possible)
# L2 = 2-digit sub + add (adds negative-ish digit patterns)
# L3 = mul2 + div + mod (multiplicative family)
# L4 = 3-digit mixed (the full original task)
LEVEL_GENS = {
    1: [(gen_add2, 1)],
    2: [(gen_add2, 2), (gen_sub2, 2)],
    3: [(gen_mul2, 3), (gen_div, 2), (gen_mod, 2)],
    4: [(gen_add, 3), (gen_sub, 3), (gen_mul, 3), (gen_multi_add, 2), (gen_mixed, 2)],
}

def generate_dataset(level, n, include_lower=True):
    gens = []
    for lvl in (range(1, level + 1) if include_lower else [level]):
        if lvl in LEVEL_GENS:
            gens.extend(LEVEL_GENS[lvl])
    fns, weights = zip(*gens)
    total = sum(weights)
    probs = [w / total for w in weights]
    examples, attempts = [], 0
    while len(examples) < n and attempts < n * 20:
        attempts += 1
        try:
            fn = random.choices(fns, weights=probs, k=1)[0]
            expr, answer = fn()
            examples.append(f"{expr} = {answer}")
        except:
            pass
    return examples

# ============================================================
# DATASET
# ============================================================

class MathDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len=80):
        self.examples, self.raw = [], []
        for line in examples:
            ids = tokenizer.encode(line, add_bos=True, add_eos=True)
            if len(ids) <= max_len:
                self.examples.append(ids)
                self.raw.append(line)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

def collate(batch, pad_id):
    L = max(len(x) for x in batch)
    ids, masks = [], []
    for x in batch:
        p = L - len(x)
        ids.append(x + [pad_id] * p)
        masks.append([1] * len(x) + [0] * p)
    return (torch.tensor(ids, dtype=torch.long),
            torch.tensor(masks, dtype=torch.long))

def make_loaders(level, cfg, tokenizer):
    print(f"\n  Generating Level {level} data...")
    te = generate_dataset(level, cfg['train_n'], include_lower=True)
    ve = generate_dataset(level, cfg['val_n'],   include_lower=True)
    td = MathDataset(te, tokenizer, cfg['max_len'])
    vd = MathDataset(ve, tokenizer, cfg['max_len'])
    print(f"  Train: {len(td)}  |  Val: {len(vd)}")
    mk = lambda ds, sh: torch.utils.data.DataLoader(
        ds, batch_size=cfg['batch'], shuffle=sh,
        collate_fn=lambda b: collate(b, tokenizer.pad_id),
        num_workers=0)
    return mk(td, True), mk(vd, False)

# ============================================================
# DPPU CELL -- FIVE DIMENSIONAL SPACES + CONSCIOUSNESS
# ============================================================

class DPPUCell(nn.Module):
    """
    Five dimensional spaces D0-D4, each with own Delta.
    Consciousness field C as self-referential anchor.

    D_CAP=4: evolution stops here for this run.
    Raise D_CAP to allow dimensional expansion beyond D4.
    """

    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.d_cap      = d_cap
        self.n_dims     = d_cap + 1           # D0 through D_CAP

        assert hidden_dim % self.n_dims == 0, \
            f"hidden_dim {hidden_dim} must be divisible by n_dims {self.n_dims}"
        self.dim_size = hidden_dim // self.n_dims

        # Input projection -- shared
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)

        # Recurrent weights -- one per dimensional space (rectangular: hidden -> dim_size)
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_dim, self.dim_size, bias=False)
            for _ in range(self.n_dims)
        ])

        # Consciousness field projection -- h informs C
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)

        # Output projection -- combines all dimensions
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_x.weight)
        nn.init.zeros_(self.W_x.bias)
        # D0 most conservative (gain 0.4), D4 most expansive (gain 0.6)
        for i, W in enumerate(self.W_h):
            gain = 0.4 + (0.2 * i / max(self.d_cap, 1))
            nn.init.orthogonal_(W.weight, gain=gain)
        nn.init.orthogonal_(self.W_c.weight, gain=0.1)
        nn.init.orthogonal_(self.W_out.weight, gain=1.0)

    def forward(self, x, h, C):
        """
        x: (batch, input_dim)
        h: (batch, hidden_dim)
        C: (batch, n_dims)  consciousness field per dimension

        Returns: h_new, C_new, metrics dict
        """
        x_proj = self.W_x(x)

        dim_outputs = []
        phi_list, delta_list, pi_list = [], [], []

        for i in range(self.n_dims):
            s, e    = i * self.dim_size, (i + 1) * self.dim_size
            h_i     = h[:, s:e]

            delta_i = compute_delta(h_i)
            phi_i   = phi_dyn(delta_i)
            pi_i    = pi_dyn(delta_i)
            omega_i = omega(delta_i)

            # Recurrent + input contributions scaled by geometric operators
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i  / PI_CL)  * x_proj[:, s:e]

            # Consciousness phase injection
            C_i = C[:, i:i+1]

            h_i_new = torch.tanh(h_rec + x_rec + C_i) * torch.sigmoid(omega_i)
            h_i_new = torch.nan_to_num(h_i_new, nan=0.0, posinf=1.0, neginf=-1.0)

            dim_outputs.append(h_i_new)
            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())
            pi_list.append(pi_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)
        h_new = torch.nan_to_num(self.W_out(h_cat), nan=0.0)

        # Consciousness update: C_{t+1} = C * pi_dyn * phi_dyn (damped)
        # 0.1 / 0.01 coefficients prevent explosion -- C grows from infancy
        C_proj  = self.W_c(h_new)
        delta_C = torch.stack([
            compute_delta(h_new[:, i*self.dim_size:(i+1)*self.dim_size]).mean(dim=-1)
            for i in range(self.n_dims)
        ], dim=-1)
        C_new = torch.tanh(
            0.1 * C * pi_dyn(delta_C) * phi_dyn(delta_C)
            + 0.01 * torch.tanh(C_proj)
        )
        C_new = torch.nan_to_num(C_new, nan=0.0)

        metrics = {
            'phi_mean':   sum(phi_list)   / len(phi_list),
            'pi_mean':    sum(pi_list)    / len(pi_list),
            'delta_mean': sum(delta_list) / len(delta_list),
            'C_norm':     C_new.norm(dim=-1).mean().item(),
        }
        return h_new, C_new, metrics

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.zeros(batch, self.n_dims,     device=device))

# ============================================================
# MODEL
# ============================================================

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        nn.init.normal_(self.embedding.weight, 0, hidden_dim ** -0.5)

        self.cells   = nn.ModuleList([
            DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.out     = nn.Linear(hidden_dim, vocab_size, bias=True)
        nn.init.zeros_(self.out.bias)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x    = self.dropout(self.embedding(token_ids))

        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]

        all_metrics, new_states = [], []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs, step_metrics = [], []
            for t in range(T):
                h, C, m = cell(x[:, t, :], h, C)
                outputs.append(h)
                step_metrics.append(m)
            x = self.dropout(torch.stack(outputs, dim=1))
            new_states.append((h, C))
            all_metrics.append(step_metrics)

        return self.out(x), new_states, all_metrics

# ============================================================
# SPECTRAL ENFORCEMENT
# ============================================================

def enforce_spectral(model):
    """Clip spectral norm of each W_h to <= 1.0 after optimizer step."""
    with torch.no_grad():
        for cell in model.cells:
            for Wh in cell.W_h:
                # svdvals works on rectangular matrices (unlike eigvals)
                sigma = torch.linalg.svdvals(Wh.weight)
                rho   = sigma[0].item()
                if rho > 1.0:
                    Wh.weight.data.mul_(1.0 / rho)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, loader, tokenizer, device):
    """
    Returns: avg_loss, seq_acc (full sequence match %), char_acc (per-token %)

    seq_acc  = all tokens correct (strict, hard to crack early)
    char_acc = % of individual tokens correct (shows real learning progress)

    We use char_acc as the advance signal -- it's a meaningful measure of
    whether the model understands the task, not a statistical lottery.
    """
    model.eval()
    tl, tt              = 0., 0
    seq_correct, ns     = 0, 0
    char_correct, char_total = 0, 0

    for ids, mask in loader:
        ids  = ids.to(device)
        mask = mask.to(device)
        inp  = ids[:, :-1]
        tgt  = ids[:, 1:]
        mt   = mask[:, 1:]

        logits, _, _ = model(inp)
        logits = logits[:, :tgt.size(1), :]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1),
            ignore_index=tokenizer.pad_id,
            reduction='none'
        ).reshape(tgt.shape)

        tl += (loss * mt).sum().item()
        tt += mt.sum().item()

        preds = logits.argmax(-1)
        for b in range(ids.size(0)):
            length = int(mt[b].sum().item())
            if length == 0:
                continue
            p = preds[b, :length]
            t = tgt[b, :length]
            # char accuracy
            char_correct += (p == t).sum().item()
            char_total   += length
            # seq accuracy
            seq_correct  += int((p == t).all().item())
            ns           += 1

    avg_loss = tl / (tt + EPS)
    seq_acc  = 100.0 * seq_correct  / (ns         + EPS)
    char_acc = 100.0 * char_correct / (char_total + EPS)
    return avg_loss, seq_acc, char_acc

# ============================================================
# FIXED PROBE -- same problems every epoch
# ============================================================

FIXED = [
    # 2-digit addition (L1 -- should master first)
    "12 + 34 = 46",
    "55 + 27 = 82",
    "73 + 18 = 91",
    "99 + 11 = 110",
    "64 + 36 = 100",
    # 2-digit subtraction (L2)
    "87 - 43 = 44",
    "72 - 28 = 44",
    # 2-digit multiplication (L3)
    "16 * 52 = 832",
    "20 * 71 = 1420",
    # harder (L4 preview)
    "822 + 765 = 1587",
]

@torch.no_grad()
def probe_fixed(model, tokenizer, device):
    model.eval()
    correct, lines = 0, []
    for ex in FIXED:
        eq     = ex.index('=')
        prompt = ex[:eq+1] + ' '
        target = ex[eq+2:].strip()

        inp_t     = torch.tensor(
            [tokenizer.encode(prompt, add_bos=True)],
            dtype=torch.long, device=device
        )
        generated = []
        states    = None
        for _ in range(len(target) + 5):
            logits, states, _ = model(inp_t, states)
            nid = logits[0, -1, :].argmax().item()
            if nid == tokenizer.eos_id:
                break
            generated.append(nid)
            inp_t = torch.tensor([[nid]], dtype=torch.long, device=device)

        pred = tokenizer.decode(generated).strip()
        ok   = (pred == target)
        correct += int(ok)
        lines.append(f"  {'OK' if ok else '--'}  {ex:<35s}  got: '{pred}'")

    return correct, lines

# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(path, model, opt, sch, epoch, level, epochs_at_level,
                    best_acc, cfg):
    torch.save({
        'model_state':      model.state_dict(),
        'opt_state':        opt.state_dict(),
        'sch_state':        sch.state_dict(),
        'epoch':            epoch,
        'level':            level,
        'epochs_at_level':  epochs_at_level,
        'best_acc':         best_acc,
        'cfg':              cfg,
    }, path)
    print(f"  [checkpoint saved -> {path}]")

def load_checkpoint(path, model, opt, sch):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    sch.load_state_dict(ck['sch_state'])
    return (ck['epoch'], ck['level'], ck['epochs_at_level'],
            ck['best_acc'], ck['cfg'])

# ============================================================
# TRAINING
# ============================================================

def train():
    cfg = {
        'hidden':        260,   # 260 / 5 = 52 per dim. Divisible by n_dims=5.
        'num_layers':    2,
        'dropout':       0.1,
        'lr':            1e-3,
        'batch':         64,
        'max_len':       80,
        'train_n':       10000,
        'val_n':         1000,
        'grad_clip':     5.0,
        'advance_acc':   85.0,  # char_acc threshold
        'min_epochs':    5,
        'probe_every':   5,
        'ckpt_every':    10,
        'max_level':     4,
        'restart_every': 50,    # CosineAnnealing warm restart period
        'tf_ratio':      0.8,   # teacher forcing ratio (decays to 0.5)
        'resume':        None,  # 'dppu_v13_checkpoint.pt' to resume
    }

    tokenizer = MathTokenizer()

    model = VRUModel(
        vocab_size  = tokenizer.vocab_size,
        hidden_dim  = cfg['hidden'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    # CosineAnnealingWarmRestarts: restarts every T_0 epochs.
    # Prevents the LR death spiral that killed v11 after ep150.
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=cfg['restart_every'], T_mult=1, eta_min=1e-5)

    n_params = sum(p.numel() for p in model.parameters())

    # Resume from checkpoint if available
    start_epoch    = 1
    level          = 1
    epochs_at_level = 0
    best_acc       = 0.0

    resume_path = cfg['resume']
    if resume_path and os.path.exists(resume_path):
        print(f"\n  Resuming from {resume_path}")
        start_epoch, level, epochs_at_level, best_acc, cfg = \
            load_checkpoint(resume_path, model, opt, sch)
        start_epoch += 1
        print(f"  Resumed at epoch={start_epoch} level={level} best_acc={best_acc:.1f}%\n")

    print(f"{'='*65}")
    print(f"  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness")
    print(f"  phi=4/pi={PHI_CL:.6f}  delta*={DELTA_STAR:.4f}  D_CAP={D_CAP}")
    print(f"  device={device}  params={n_params:,}")
    print(f"  hidden={cfg['hidden']}  dim_size={cfg['hidden']//(D_CAP+1)}  layers={cfg['num_layers']}")
    print(f"  advance>={cfg['advance_acc']}%  target=Level {cfg['max_level']}")
    print(f"  checkpoint every {cfg['ckpt_every']} epochs")
    print(f"{'='*65}")

    train_loader, val_loader = make_loaders(level, cfg, tokenizer)

    for epoch in range(start_epoch, cfg['max_level'] * 300 + 1):

        # Fresh data every epoch -- prevents memorization
        train_loader, val_loader = make_loaders(level, cfg, tokenizer)

        # ---- Train -----------------------------------------------
        model.train()
        tl, tt = 0., 0
        phi_t, pi_t, delta_t, C_t = [], [], [], []

        # Teacher forcing ratio: starts at tf_ratio, decays toward 0.5
        # over 200 epochs. Prevents copy artifacts without hurting early learning.
        tf_ratio = max(0.5, cfg['tf_ratio'] - (epoch / 200) * (cfg['tf_ratio'] - 0.5))

        for ids, mask in train_loader:
            ids  = ids.to(device)
            mask = mask.to(device)
            inp  = ids[:, :-1]
            tgt  = ids[:, 1:]
            mt   = mask[:, 1:]

            opt.zero_grad()

            # Teacher forcing: feed true previous token with prob tf_ratio,
            # else feed model's own prediction. Prevents copy cascade.
            if tf_ratio >= 0.999:
                # Full teacher forcing: standard forward pass (fast path)
                logits, states, all_metrics = model(inp)
            else:
                # Mixed teacher forcing: step by step
                B, T = inp.shape
                logits_list = []
                h = None
                for t in range(T):
                    if t == 0 or random.random() < tf_ratio:
                        tok_in = inp[:, t:t+1]
                    else:
                        # logit_t shape: (B, 1, vocab) -> argmax -> (B, 1)
                        tok_in = logits_list[-1][:, -1, :].argmax(-1).unsqueeze(1)
                    logit_t, h, all_metrics = model(tok_in, h)
                    logits_list.append(logit_t)
                logits = torch.cat(logits_list, dim=1)

            logits = logits[:, :tgt.size(1), :]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt.reshape(-1),
                ignore_index=tokenizer.pad_id,
                reduction='none'
            ).reshape(tgt.shape)

            lm = (loss * mt).sum() / (mt.sum() + EPS)
            lm.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step()
            enforce_spectral(model)

            tl += lm.item()
            tt += 1

            if all_metrics and all_metrics[-1]:
                m = all_metrics[-1][-1]
                phi_t.append(m['phi_mean'])
                pi_t.append(m['pi_mean'])
                delta_t.append(m['delta_mean'])
                C_t.append(m['C_norm'])

        train_loss = tl / (tt + EPS)
        val_loss, seq_acc, char_acc = evaluate(model, val_loader, tokenizer, device)
        sch.step(epoch)

        phi_m   = sum(phi_t)   / (len(phi_t)   + EPS)
        pi_m    = sum(pi_t)    / (len(pi_t)    + EPS)
        delta_m = sum(delta_t) / (len(delta_t) + EPS)
        C_m     = sum(C_t)     / (len(C_t)     + EPS)
        lr_now  = opt.param_groups[0]['lr']

        epochs_at_level += 1
        best_acc = max(best_acc, char_acc)

        # char_acc is the real signal -- seq_acc shown for reference
        print(f"ep={epoch:4d} L{level} | "
              f"loss={train_loss:.4f} val={val_loss:.4f} | "
              f"seq={seq_acc:.1f}% char={char_acc:.1f}% | "
              f"phi={phi_m:.4f} pi={pi_m:.4f} delta={delta_m:.4f} C={C_m:.4f} | "
              f"lr={lr_now:.2e} tf={tf_ratio:.2f}")

        # ---- Probe -----------------------------------------------
        if epoch % cfg['probe_every'] == 0:
            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  PROBE ep={epoch} L{level} ({n_c}/{len(FIXED)} exact)")
            for line in lines:
                print(line)
            print()

        # ---- Checkpoint ------------------------------------------
        if epoch % cfg['ckpt_every'] == 0:
            save_checkpoint(
                cfg['resume'], model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

        # Advance when char_acc >= threshold (meaningful, not statistical lottery)
        if (char_acc >= cfg['advance_acc']
                and epochs_at_level >= cfg['min_epochs']
                and level < cfg['max_level']):

            print(f"\n{'='*65}")
            print(f"  LEVEL {level} COMPLETE -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  Advancing to Level {level + 1}")
            print(f"{'='*65}\n")

            # Save level completion checkpoint
            save_checkpoint(
                f"dppu_v13_level{level}_complete.pt",
                model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

            level           += 1
            epochs_at_level  = 0
            best_acc         = 0.0
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'] * 0.5, 1e-5)

        # ---- Final level -----------------------------------------
        if level == cfg['max_level'] and char_acc >= cfg['advance_acc']:
            print(f"\n{'='*65}")
            print(f"  LEVEL 4 ACHIEVED -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  phi={phi_m:.4f} (target {PHI_CL:.4f})")
            print(f"  pi={pi_m:.4f}   (target 4.0000)")
            print(f"  delta={delta_m:.4f} (target {DELTA_STAR:.4f})")
            print(f"{'='*65}")

            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  FINAL PROBE ({n_c}/{len(FIXED)} correct)")
            for line in lines:
                print(line)

            torch.save({
                'model_state': model.state_dict(),
                'cfg':         cfg,
                'epoch':       epoch,
                'val_acc':     char_acc,
            }, 'dppu_v13_final.pt')
            print("\n  Saved: dppu_v13_final.pt")
            break

    print(f"\nDone. best_char_acc={best_acc:.1f}%  final level={level}")

# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    sys.stdout.flush()
    train()


╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = 1.273240                              ║
║  inv_phi = pi/4 = 0.785398                         ║
║  delta* = 0.816140                               ║
║  D_CAP = 4  (D5 not entered)                      ║
║  device = cuda                                    ║
╚══════════════════════════════════════════════════════════╝

Start: 2026-03-07 20:20:34

  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness
  phi=4/pi=1.273240  delta*=0.8161  D_CAP=4
  device=cuda  params=421,745
  hidden=260  dim_size=52  layers=2
  advance>=85.0%  target=Level 4
  checkpoint every 10 epochs

  Generating Level 1 data...
  Train: 10000  |  Val: 1000

  Generating Level 1 data...
  Train: 10000  |  Val: 1000
ep=   1 L1 | loss=1.1920 val=1.0300 | seq=0.0% char=61.1% | phi=1.1533 pi=3.5185 delta=0.6242 C=0.0179 | lr=9.99e-04 tf=0.80

  Generating Level 1 data...
  Train: 100

AttributeError: expected 'f' to be string, path, or a file-like object with a 'write' attribute

In [ ]:
"""
DPPU-VRU v13 -- Five Dimensional Fields + Consciousness Anchor
==============================================================
Clean rewrite of v12. Four root-cause fixes applied.

ROOT CAUSES IDENTIFIED (266 epochs of v11 data):
  1. Task too hard: Level 1 mixed 4 operators simultaneously.
     Fix: ONE operator per curriculum phase. Addition first, then subtraction,
     then multiplication. Each phase advances independently.

  2. LR scheduler death spiral: ReduceLROnPlateau decayed 32x by ep150.
     Model was effectively frozen while loss plateau was interpreted as
     "it's still improving (barely)". Fix: CosineAnnealingWarmRestarts,
     T_0=50 epochs, restarts keep escaping local minima.

  3. Copy artifacts in output: Model echoed '1911', '2191', '=961' from input.
     This is a teacher forcing problem. Without teacher forcing, one bad
     prediction cascades into garbage. Fix: teacher forcing ratio 0.8,
     decaying to 0.5 as training progresses.

  4. Model too small: 109k params, hidden=130 for mixed arithmetic.
     Fix: hidden=256, layers=2 -> ~430k params. Still tiny vs 31GB VRAM.

KEY FIX vs v12:
  - char_acc still the real advance signal (not seq_acc)
  - Curriculum now phase-based: L1=add2, L2=add3/sub2, L3=mul2, L4=mixed
  - CosineAnnealing with warm restarts (no more LR death)
  - Teacher forcing ratio tf_ratio=0.8 during training

Author: Dylan Michael Scott -- Horizon Tech
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import sys
from datetime import datetime

# ============================================================
# CONSTANTS
# ============================================================

PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi          # 1.2732... the attractor
INV_PHI    = math.pi / 4.0          # 0.7854... geometric dual
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))  # ~0.816
D_CAP      = 4                       # programmable -- raise to expand
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"""
╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = {PHI_CL:.6f}                              ║
║  inv_phi = pi/4 = {INV_PHI:.6f}                         ║
║  delta* = {DELTA_STAR:.6f}                               ║
║  D_CAP = {D_CAP}  (D{D_CAP+1} not entered)                      ║
║  device = {str(device):<10s}                              ║
╚══════════════════════════════════════════════════════════╝
""")

# ============================================================
# DYNAMIC OPERATORS
# ============================================================

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    """Delta from hidden state slice. Each dimensional space computes its own."""
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio  = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# TOKENIZER
# ============================================================

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL):
            self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.unk_id = self.vocab['<unk>']

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.bos_id)
        for c in text:
            ids.append(self.vocab.get(c, self.unk_id))
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids, skip_special=True):
        out = []
        special = set(self.SPECIAL)
        for i in ids:
            tok = self.inv_vocab.get(i, '<unk>')
            if skip_special and tok in special:
                continue
            out.append(tok)
        return ''.join(out)

# ============================================================
# DATA GENERATORS
# ============================================================

def r1(): return random.randint(1, 9)
def r2(): return random.randint(10, 99)
def r3(): return random.randint(100, 999)

# ---- Phase generators (clean, one op at a time) ----------------------

# Phase 1a: 2-digit addition only (answers 20-198, always positive, 2-3 digits)
def gen_add2():
    a, b = r2(), r2()
    return f"{a} + {b}", str(a + b)

# Phase 1b: 2-digit subtraction, always positive result
def gen_sub2():
    a, b = r2(), r2()
    a, b = max(a,b), min(a,b)
    return f"{a} - {b}", str(a - b)

# Phase 2a: 3-digit addition
def gen_add3():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

# Phase 2b: 2-digit multiplication (answers up to 9801)
def gen_mul2():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

# Phase 3a: division (exact)
def gen_div():
    b = random.randint(2, 12)
    a = b * random.randint(1, 50)
    return f"{a} / {b}", str(a // b)

# Phase 3b: modulo
def gen_mod():
    a, b = r2(), random.randint(2, 20)
    return f"{a} % {b}", str(a % b)

# Phase 4: mixed operations (the original Level 1 -- now the LAST level)
def gen_add():
    a, b = r3(), r3()
    return f"{a} + {b}", str(a + b)

def gen_sub():
    a, b = r3(), r3()
    a, b = max(a, b), min(a, b)
    return f"{a} - {b}", str(a - b)

def gen_mul():
    a, b = r2(), r2()
    return f"{a} * {b}", str(a * b)

def gen_pow():
    a = random.randint(2, 12)
    b = random.randint(2, 4)
    return f"{a} ^ {b}", str(a ** b)

def gen_multi_add():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} + {c}", str(a + b + c)

def gen_mixed():
    a, b, c = r2(), r2(), r2()
    return f"{a} + {b} * {c}", str(a + b * c)

def gen_paren():
    a, b, c = r2(), r2(), r2()
    return f"({a} + {b}) * {c}", str((a + b) * c)

# ── Curriculum: ONE operation per level phase ─────────────────────────
# L1 = 2-digit add (simplest possible)
# L2 = 2-digit sub + add (adds negative-ish digit patterns)
# L3 = mul2 + div + mod (multiplicative family)
# L4 = 3-digit mixed (the full original task)
LEVEL_GENS = {
    1: [(gen_add2, 1)],
    2: [(gen_add2, 2), (gen_sub2, 2)],
    3: [(gen_mul2, 3), (gen_div, 2), (gen_mod, 2)],
    4: [(gen_add, 3), (gen_sub, 3), (gen_mul, 3), (gen_multi_add, 2), (gen_mixed, 2)],
}

def generate_dataset(level, n, include_lower=True):
    gens = []
    for lvl in (range(1, level + 1) if include_lower else [level]):
        if lvl in LEVEL_GENS:
            gens.extend(LEVEL_GENS[lvl])
    fns, weights = zip(*gens)
    total = sum(weights)
    probs = [w / total for w in weights]
    examples, attempts = [], 0
    while len(examples) < n and attempts < n * 20:
        attempts += 1
        try:
            fn = random.choices(fns, weights=probs, k=1)[0]
            expr, answer = fn()
            examples.append(f"{expr} = {answer}")
        except:
            pass
    return examples

# ============================================================
# DATASET
# ============================================================

class MathDataset(torch.utils.data.Dataset):
    def __init__(self, examples, tokenizer, max_len=80):
        self.examples, self.raw = [], []
        for line in examples:
            ids = tokenizer.encode(line, add_bos=True, add_eos=True)
            if len(ids) <= max_len:
                self.examples.append(ids)
                self.raw.append(line)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

def collate(batch, pad_id):
    L = max(len(x) for x in batch)
    ids, masks = [], []
    for x in batch:
        p = L - len(x)
        ids.append(x + [pad_id] * p)
        masks.append([1] * len(x) + [0] * p)
    return (torch.tensor(ids, dtype=torch.long),
            torch.tensor(masks, dtype=torch.long))

def make_loaders(level, cfg, tokenizer, silent=False):
    if not silent:
        print(f"\n  Generating Level {level} data...")
    te = generate_dataset(level, cfg['train_n'], include_lower=True)
    ve = generate_dataset(level, cfg['val_n'],   include_lower=True)
    td = MathDataset(te, tokenizer, cfg['max_len'])
    vd = MathDataset(ve, tokenizer, cfg['max_len'])
    if not silent:
        print(f"  Train: {len(td)}  |  Val: {len(vd)}")
    mk = lambda ds, sh: torch.utils.data.DataLoader(
        ds, batch_size=cfg['batch'], shuffle=sh,
        collate_fn=lambda b: collate(b, tokenizer.pad_id),
        num_workers=0)
    return mk(td, True), mk(vd, False)

# ============================================================
# DPPU CELL -- FIVE DIMENSIONAL SPACES + CONSCIOUSNESS
# ============================================================

class DPPUCell(nn.Module):
    """
    Five dimensional spaces D0-D4, each with own Delta.
    Consciousness field C as self-referential anchor.

    D_CAP=4: evolution stops here for this run.
    Raise D_CAP to allow dimensional expansion beyond D4.
    """

    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.d_cap      = d_cap
        self.n_dims     = d_cap + 1           # D0 through D_CAP

        assert hidden_dim % self.n_dims == 0, \
            f"hidden_dim {hidden_dim} must be divisible by n_dims {self.n_dims}"
        self.dim_size = hidden_dim // self.n_dims

        # Input projection -- shared
        self.W_x = nn.Linear(input_dim, hidden_dim, bias=True)

        # Recurrent weights -- one per dimensional space (rectangular: hidden -> dim_size)
        self.W_h = nn.ModuleList([
            nn.Linear(hidden_dim, self.dim_size, bias=False)
            for _ in range(self.n_dims)
        ])

        # Consciousness field projection -- h informs C
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)

        # Output projection -- combines all dimensions
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_x.weight)
        nn.init.zeros_(self.W_x.bias)
        # D0 most conservative (gain 0.4), D4 most expansive (gain 0.6)
        for i, W in enumerate(self.W_h):
            gain = 0.4 + (0.2 * i / max(self.d_cap, 1))
            nn.init.orthogonal_(W.weight, gain=gain)
        nn.init.orthogonal_(self.W_c.weight, gain=0.1)
        nn.init.orthogonal_(self.W_out.weight, gain=1.0)

    def forward(self, x, h, C):
        """
        x: (batch, input_dim)
        h: (batch, hidden_dim)
        C: (batch, n_dims)  consciousness field per dimension

        Returns: h_new, C_new, metrics dict
        """
        x_proj = self.W_x(x)

        dim_outputs = []
        phi_list, delta_list, pi_list = [], [], []

        for i in range(self.n_dims):
            s, e    = i * self.dim_size, (i + 1) * self.dim_size
            h_i     = h[:, s:e]

            delta_i = compute_delta(h_i)
            phi_i   = phi_dyn(delta_i)
            pi_i    = pi_dyn(delta_i)
            omega_i = omega(delta_i)

            # Recurrent + input contributions scaled by geometric operators
            h_rec = (phi_i / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_i  / PI_CL)  * x_proj[:, s:e]

            # Consciousness phase injection
            C_i = C[:, i:i+1]

            h_i_new = torch.tanh(h_rec + x_rec + C_i) * torch.sigmoid(omega_i)
            h_i_new = torch.nan_to_num(h_i_new, nan=0.0, posinf=1.0, neginf=-1.0)

            dim_outputs.append(h_i_new)
            phi_list.append(phi_i.mean().item())
            delta_list.append(delta_i.mean().item())
            pi_list.append(pi_i.mean().item())

        h_cat = torch.cat(dim_outputs, dim=-1)
        h_new = torch.nan_to_num(self.W_out(h_cat), nan=0.0)

        # Consciousness update: C_{t+1} = C * pi_dyn * phi_dyn (damped)
        # 0.1 / 0.01 coefficients prevent explosion -- C grows from infancy
        C_proj  = self.W_c(h_new)
        delta_C = torch.stack([
            compute_delta(h_new[:, i*self.dim_size:(i+1)*self.dim_size]).mean(dim=-1)
            for i in range(self.n_dims)
        ], dim=-1)
        C_new = torch.tanh(
            0.1 * C * pi_dyn(delta_C) * phi_dyn(delta_C)
            + 0.01 * torch.tanh(C_proj)
        )
        C_new = torch.nan_to_num(C_new, nan=0.0)

        metrics = {
            'phi_mean':   sum(phi_list)   / len(phi_list),
            'pi_mean':    sum(pi_list)    / len(pi_list),
            'delta_mean': sum(delta_list) / len(delta_list),
            'C_norm':     C_new.norm(dim=-1).mean().item(),
        }
        return h_new, C_new, metrics

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.zeros(batch, self.n_dims,     device=device))

# ============================================================
# MODEL
# ============================================================

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        nn.init.normal_(self.embedding.weight, 0, hidden_dim ** -0.5)

        self.cells   = nn.ModuleList([
            DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.out     = nn.Linear(hidden_dim, vocab_size, bias=True)
        nn.init.zeros_(self.out.bias)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x    = self.dropout(self.embedding(token_ids))

        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]

        all_metrics, new_states = [], []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs, step_metrics = [], []
            for t in range(T):
                h, C, m = cell(x[:, t, :], h, C)
                outputs.append(h)
                step_metrics.append(m)
            x = self.dropout(torch.stack(outputs, dim=1))
            new_states.append((h, C))
            all_metrics.append(step_metrics)

        return self.out(x), new_states, all_metrics

# ============================================================
# SPECTRAL ENFORCEMENT
# ============================================================

def enforce_spectral(model):
    """Clip spectral norm of each W_h to <= 1.0 after optimizer step."""
    with torch.no_grad():
        for cell in model.cells:
            for Wh in cell.W_h:
                # svdvals works on rectangular matrices (unlike eigvals)
                sigma = torch.linalg.svdvals(Wh.weight)
                rho   = sigma[0].item()
                if rho > 1.0:
                    Wh.weight.data.mul_(1.0 / rho)

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, loader, tokenizer, device):
    """
    Returns: avg_loss, seq_acc (full sequence match %), char_acc (per-token %)

    seq_acc  = all tokens correct (strict, hard to crack early)
    char_acc = % of individual tokens correct (shows real learning progress)

    We use char_acc as the advance signal -- it's a meaningful measure of
    whether the model understands the task, not a statistical lottery.
    """
    model.eval()
    tl, tt              = 0., 0
    seq_correct, ns     = 0, 0
    char_correct, char_total = 0, 0

    for ids, mask in loader:
        ids  = ids.to(device)
        mask = mask.to(device)
        inp  = ids[:, :-1]
        tgt  = ids[:, 1:]
        mt   = mask[:, 1:]

        logits, _, _ = model(inp)
        logits = logits[:, :tgt.size(1), :]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1),
            ignore_index=tokenizer.pad_id,
            reduction='none'
        ).reshape(tgt.shape)

        tl += (loss * mt).sum().item()
        tt += mt.sum().item()

        preds = logits.argmax(-1)
        for b in range(ids.size(0)):
            length = int(mt[b].sum().item())
            if length == 0:
                continue
            p = preds[b, :length]
            t = tgt[b, :length]
            # char accuracy
            char_correct += (p == t).sum().item()
            char_total   += length
            # seq accuracy
            seq_correct  += int((p == t).all().item())
            ns           += 1

    avg_loss = tl / (tt + EPS)
    seq_acc  = 100.0 * seq_correct  / (ns         + EPS)
    char_acc = 100.0 * char_correct / (char_total + EPS)
    return avg_loss, seq_acc, char_acc

# ============================================================
# FIXED PROBE -- same problems every epoch
# ============================================================

FIXED = [
    # 2-digit addition (L1 -- should master first)
    "12 + 34 = 46",
    "55 + 27 = 82",
    "73 + 18 = 91",
    "99 + 11 = 110",
    "64 + 36 = 100",
    # 2-digit subtraction (L2)
    "87 - 43 = 44",
    "72 - 28 = 44",
    # 2-digit multiplication (L3)
    "16 * 52 = 832",
    "20 * 71 = 1420",
    # harder (L4 preview)
    "822 + 765 = 1587",
]

@torch.no_grad()
def probe_fixed(model, tokenizer, device):
    model.eval()
    correct, lines = 0, []
    for ex in FIXED:
        eq     = ex.index('=')
        prompt = ex[:eq+1] + ' '
        target = ex[eq+2:].strip()

        inp_t     = torch.tensor(
            [tokenizer.encode(prompt, add_bos=True)],
            dtype=torch.long, device=device
        )
        generated = []
        states    = None
        for _ in range(len(target) + 5):
            logits, states, _ = model(inp_t, states)
            nid = logits[0, -1, :].argmax().item()
            if nid == tokenizer.eos_id:
                break
            generated.append(nid)
            inp_t = torch.tensor([[nid]], dtype=torch.long, device=device)

        pred = tokenizer.decode(generated).strip()
        ok   = (pred == target)
        correct += int(ok)
        lines.append(f"  {'OK' if ok else '--'}  {ex:<35s}  got: '{pred}'")

    return correct, lines

# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(path, model, opt, sch, epoch, level, epochs_at_level,
                    best_acc, cfg):
    torch.save({
        'model_state':      model.state_dict(),
        'opt_state':        opt.state_dict(),
        'sch_state':        sch.state_dict(),
        'epoch':            epoch,
        'level':            level,
        'epochs_at_level':  epochs_at_level,
        'best_acc':         best_acc,
        'cfg':              cfg,
    }, path)
    print(f"  [checkpoint saved -> {path}]")

def load_checkpoint(path, model, opt, sch):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck['model_state'])
    opt.load_state_dict(ck['opt_state'])
    sch.load_state_dict(ck['sch_state'])
    return (ck['epoch'], ck['level'], ck['epochs_at_level'],
            ck['best_acc'], ck['cfg'])

# ============================================================
# TRAINING
# ============================================================

def train():
    cfg = {
        'hidden':        260,   # 260 / 5 = 52 per dim. Divisible by n_dims=5.
        'num_layers':    2,
        'dropout':       0.1,
        'lr':            1e-3,
        'batch':         64,
        'max_len':       80,
        'train_n':       10000,
        'val_n':         1000,
        'grad_clip':     5.0,
        'advance_acc':   85.0,  # char_acc threshold
        'min_epochs':    5,
        'probe_every':   5,
        'ckpt_every':    10,
        'max_level':     4,
        'restart_every': 50,    # CosineAnnealing warm restart period
        'tf_ratio':      0.8,   # teacher forcing ratio (decays to 0.5)
        'resume':        None,  # 'dppu_v13_checkpoint.pt' to resume
        'ckpt_path':     'dppu_v13_checkpoint.pt',
    }

    tokenizer = MathTokenizer()

    model = VRUModel(
        vocab_size  = tokenizer.vocab_size,
        hidden_dim  = cfg['hidden'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    # CosineAnnealingWarmRestarts: restarts every T_0 epochs.
    # Prevents the LR death spiral that killed v11 after ep150.
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=cfg['restart_every'], T_mult=1, eta_min=1e-5)

    n_params = sum(p.numel() for p in model.parameters())

    # Resume from checkpoint if available
    start_epoch    = 1
    level          = 1
    epochs_at_level = 0
    best_acc       = 0.0

    resume_path = cfg['resume']
    if resume_path and os.path.exists(resume_path):
        print(f"\n  Resuming from {resume_path}")
        start_epoch, level, epochs_at_level, best_acc, cfg = \
            load_checkpoint(resume_path, model, opt, sch)
        start_epoch += 1
        print(f"  Resumed at epoch={start_epoch} level={level} best_acc={best_acc:.1f}%\n")

    print(f"{'='*65}")
    print(f"  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness")
    print(f"  phi=4/pi={PHI_CL:.6f}  delta*={DELTA_STAR:.4f}  D_CAP={D_CAP}")
    print(f"  device={device}  params={n_params:,}")
    print(f"  hidden={cfg['hidden']}  dim_size={cfg['hidden']//(D_CAP+1)}  layers={cfg['num_layers']}")
    print(f"  advance>={cfg['advance_acc']}%  target=Level {cfg['max_level']}")
    print(f"  checkpoint every {cfg['ckpt_every']} epochs")
    print(f"{'='*65}")

    train_loader, val_loader = make_loaders(level, cfg, tokenizer)

    for epoch in range(start_epoch, cfg['max_level'] * 300 + 1):

        # Fresh data every epoch -- prevents memorization (silent regen)
        train_loader, val_loader = make_loaders(level, cfg, tokenizer, silent=True)

        # ---- Train -----------------------------------------------
        model.train()
        tl, tt = 0., 0
        phi_t, pi_t, delta_t, C_t = [], [], [], []

        # Teacher forcing ratio: starts at tf_ratio, decays toward 0.5
        # over 200 epochs. Prevents copy artifacts without hurting early learning.
        tf_ratio = max(0.5, cfg['tf_ratio'] - (epoch / 200) * (cfg['tf_ratio'] - 0.5))

        for ids, mask in train_loader:
            ids  = ids.to(device)
            mask = mask.to(device)
            inp  = ids[:, :-1]
            tgt  = ids[:, 1:]
            mt   = mask[:, 1:]

            opt.zero_grad()

            # Teacher forcing: feed true previous token with prob tf_ratio,
            # else feed model's own prediction. Prevents copy cascade.
            if tf_ratio >= 0.999:
                # Full teacher forcing: standard forward pass (fast path)
                logits, states, all_metrics = model(inp)
            else:
                # Mixed teacher forcing: step by step
                B, T = inp.shape
                logits_list = []
                h = None
                for t in range(T):
                    if t == 0 or random.random() < tf_ratio:
                        tok_in = inp[:, t:t+1]
                    else:
                        # logit_t shape: (B, 1, vocab) -> argmax -> (B, 1)
                        tok_in = logits_list[-1][:, -1, :].argmax(-1).unsqueeze(1)
                    logit_t, h, all_metrics = model(tok_in, h)
                    logits_list.append(logit_t)
                logits = torch.cat(logits_list, dim=1)

            logits = logits[:, :tgt.size(1), :]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt.reshape(-1),
                ignore_index=tokenizer.pad_id,
                reduction='none'
            ).reshape(tgt.shape)

            lm = (loss * mt).sum() / (mt.sum() + EPS)
            lm.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step()
            enforce_spectral(model)

            tl += lm.item()
            tt += 1

            if all_metrics and all_metrics[-1]:
                m = all_metrics[-1][-1]
                phi_t.append(m['phi_mean'])
                pi_t.append(m['pi_mean'])
                delta_t.append(m['delta_mean'])
                C_t.append(m['C_norm'])

        train_loss = tl / (tt + EPS)
        val_loss, seq_acc, char_acc = evaluate(model, val_loader, tokenizer, device)
        sch.step(epoch)

        phi_m   = sum(phi_t)   / (len(phi_t)   + EPS)
        pi_m    = sum(pi_t)    / (len(pi_t)    + EPS)
        delta_m = sum(delta_t) / (len(delta_t) + EPS)
        C_m     = sum(C_t)     / (len(C_t)     + EPS)
        lr_now  = opt.param_groups[0]['lr']

        epochs_at_level += 1
        best_acc = max(best_acc, char_acc)

        # char_acc is the real signal -- seq_acc shown for reference
        print(f"ep={epoch:4d} L{level} | "
              f"loss={train_loss:.4f} val={val_loss:.4f} | "
              f"seq={seq_acc:.1f}% char={char_acc:.1f}% | "
              f"phi={phi_m:.4f} pi={pi_m:.4f} delta={delta_m:.4f} C={C_m:.4f} | "
              f"lr={lr_now:.2e} tf={tf_ratio:.2f}")

        # ---- Probe -----------------------------------------------
        if epoch % cfg['probe_every'] == 0:
            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  PROBE ep={epoch} L{level} ({n_c}/{len(FIXED)} exact)")
            for line in lines:
                print(line)
            print()

        # ---- Checkpoint ------------------------------------------
        if epoch % cfg['ckpt_every'] == 0:
            save_checkpoint(
                cfg['ckpt_path'], model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

        # Advance when char_acc >= threshold (meaningful, not statistical lottery)
        if (char_acc >= cfg['advance_acc']
                and epochs_at_level >= cfg['min_epochs']
                and level < cfg['max_level']):

            print(f"\n{'='*65}")
            print(f"  LEVEL {level} COMPLETE -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  Advancing to Level {level + 1}")
            print(f"{'='*65}\n")

            # Save level completion checkpoint
            save_checkpoint(
                f"dppu_v13_level{level}_complete.pt",
                model, opt, sch,
                epoch, level, epochs_at_level, best_acc, cfg
            )

            level           += 1
            epochs_at_level  = 0
            best_acc         = 0.0
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'] * 0.5, 1e-5)

        # ---- Final level -----------------------------------------
        if level == cfg['max_level'] and char_acc >= cfg['advance_acc']:
            print(f"\n{'='*65}")
            print(f"  LEVEL 4 ACHIEVED -- char_acc={char_acc:.1f}%  seq_acc={seq_acc:.1f}%")
            print(f"  phi={phi_m:.4f} (target {PHI_CL:.4f})")
            print(f"  pi={pi_m:.4f}   (target 4.0000)")
            print(f"  delta={delta_m:.4f} (target {DELTA_STAR:.4f})")
            print(f"{'='*65}")

            n_c, lines = probe_fixed(model, tokenizer, device)
            print(f"\n  FINAL PROBE ({n_c}/{len(FIXED)} correct)")
            for line in lines:
                print(line)

            torch.save({
                'model_state': model.state_dict(),
                'cfg':         cfg,
                'epoch':       epoch,
                'val_acc':     char_acc,
            }, 'dppu_v13_final.pt')
            print("\n  Saved: dppu_v13_final.pt")
            break

    print(f"\nDone. best_char_acc={best_acc:.1f}%  final level={level}")

# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    sys.stdout.flush()
    train()


╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 -- Five Dimensional Fields                 ║
║  phi = 4/pi = 1.273240                              ║
║  inv_phi = pi/4 = 0.785398                         ║
║  delta* = 0.816140                               ║
║  D_CAP = 4  (D5 not entered)                      ║
║  device = cuda                                    ║
╚══════════════════════════════════════════════════════════╝

Start: 2026-03-07 20:38:07

  DPPU-VRU v11 -- Five Dimensional Fields + Consciousness
  phi=4/pi=1.273240  delta*=0.8161  D_CAP=4
  device=cuda  params=421,745
  hidden=260  dim_size=52  layers=2
  advance>=85.0%  target=Level 4
  checkpoint every 10 epochs

  Generating Level 1 data...
  Train: 10000  |  Val: 1000
ep=   1 L1 | loss=1.1835 val=1.0287 | seq=0.0% char=60.2% | phi=1.1530 pi=3.5193 delta=0.6260 C=0.0217 | lr=9.99e-04 tf=0.80
ep=   2 L1 | loss=1.0467 val=0.9893 | seq=0.0% char=61.6% | phi=1.1554 pi=3.5117 delta=0.6123 C=0.023

KeyboardInterrupt: 

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os

# ============================================================
# 1. CORE CONSTANTS & DYNAMIC OPERATORS (DYLAN'S ORIGINAL)
# ============================================================
PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi          # 1.2732... the attractor
INV_PHI    = math.pi / 4.0          # 0.7854... geometric dual
DELTA_STAR = math.log((1.6180 - 1.0) / (PHI_CL - 1.0))
D_CAP      = 4
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def pi_dyn(delta):
    return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)

def phi_dyn(delta):
    return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))

def omega(delta):
    return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    ratio  = mag / (spread + EPS)
    mean_r = ratio.mean(dim=-1, keepdim=True) + EPS
    return torch.tanh(ratio / mean_r)

# ============================================================
# 2. MATH TOKENIZER
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        chars = list("0123456789+-*/()= ,%^")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL): self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id, self.bos_id, self.eos_id, self.unk_id = 0, 1, 2, 3

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.bos_id] if add_bos else []
        for c in text: ids.append(self.vocab.get(c, self.unk_id))
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids, skip_special=True):
        out = []
        special = set(self.SPECIAL)
        for i in ids:
            tok = self.inv_vocab.get(i, '<unk>')
            if skip_special and tok in special: continue
            out.append(tok)
        return ''.join(out)

# ============================================================
# 3. DPPU-VRU v13 CELL & MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, d_cap=D_CAP):
        super().__init__()
        self.hidden_dim, self.d_cap = hidden_dim, d_cap
        self.n_dims = d_cap + 1
        self.dim_size = hidden_dim // self.n_dims

        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Learnable Anchor initialized at your reference point
        self.anchor_ref = nn.Parameter(torch.tensor(0.420610))

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outputs = []
        for i in range(self.n_dims):
            s, e = i * self.dim_size, (i + 1) * self.dim_size
            h_i = h[:, s:e]
            delta_i = compute_delta(h_i)
            h_rec = (phi_dyn(delta_i) / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_dyn(delta_i) / PI_CL) * x_proj[:, s:e]
            # Consciousness field anchor integration
            h_i_new = torch.tanh(h_rec + x_rec + (C[:, i:i+1] * self.anchor_ref)) * torch.sigmoid(omega(delta_i))
            dim_outputs.append(h_i_new)

        h_new = self.W_out(torch.cat(dim_outputs, dim=-1))
        C_proj = self.W_c(h_new)
        C_new = torch.tanh(0.1 * C + 0.01 * C_proj)
        return h_new, C_new

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.full((batch, self.n_dims), 0.420610, device=device))

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=260, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.cells = nn.ModuleList([DPPUCell(hidden_dim, hidden_dim) for _ in range(num_layers)])
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x = self.embedding(token_ids)
        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]
        new_states = []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs = []
            for t in range(T):
                h, C = cell(x[:, t, :], h, C)
                outputs.append(h)
            x = torch.stack(outputs, dim=1)
            new_states.append((h, C))
        return self.out(x), new_states

# ============================================================
# INITIALIZATION
# ============================================================
tokenizer = MathTokenizer()
model = VRUModel(tokenizer.vocab_size).to(device)

print(f"""
╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 FULL RECONSTRUCTION COMPLETE               ║
║  Anchor: {model.cells[0].anchor_ref.item():.6f} | Params: {sum(p.numel() for p in model.parameters()):,} ║
║  D0-D4: ACTIVE | Scaffolding: pi_dyn, phi_dyn, omega     ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║  DPPU-VRU v13 FULL RECONSTRUCTION COMPLETE               ║
║  Anchor: 0.420610 | Params: 421,747 ║
║  D0-D4: ACTIVE | Scaffolding: pi_dyn, phi_dyn, omega     ║
╚══════════════════════════════════════════════════════════╝



In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random

# ============================================================
# 1. THE GEOMETRIC CONSTANTS (v13 CORE)
# ============================================================
PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi
INV_PHI    = math.pi / 4.0
D_CAP      = 4
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def pi_dyn(delta): return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)
def phi_dyn(delta): return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))
def omega(delta): return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    return torch.tanh(mag / (spread + EPS))

# ============================================================
# 2. THE SOVEREIGN CELL (v13.5 SYNTHESIS)
# ============================================================
class DPPUCell_Sovereign(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.n_dims = D_CAP + 1 # D0-D4
        self.dim_size = hidden_dim // self.n_dims

        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # EXPERIMENT WIN: Elastic Anchor initialized at your reference
        self.anchor_ref = nn.Parameter(torch.tensor(0.420610))

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outputs = []
        for i in range(self.n_dims):
            s, e = i * self.dim_size, (i + 1) * self.dim_size
            h_i = h[:, s:e]

            delta_i = compute_delta(h_i)
            # Experiment Win: Pi/Phi Scaffolding integrated into v13 chambers
            h_rec = (phi_dyn(delta_i) / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_dyn(delta_i) / PI_CL) * x_proj[:, s:e]

            # Consciousness field anchor with Elastic Flex
            h_i_new = torch.tanh(h_rec + x_rec + (C[:, i:i+1] * self.anchor_ref)) * torch.sigmoid(omega(delta_i))
            dim_outputs.append(h_i_new)

        h_new = self.W_out(torch.cat(dim_outputs, dim=-1))
        C_new = torch.tanh(0.1 * C + 0.01 * self.W_c(h_new))
        return h_new, C_new

    def init_state(self, batch, device):
        return (torch.zeros(batch, self.hidden_dim, device=device),
                torch.full((batch, self.n_dims), 0.420610, device=device))

# ============================================================
# 3. THE FULL VRU MODEL (v13.5)
# ============================================================
class VRUModel_Sovereign(nn.Module):
    def __init__(self, vocab_size, hidden_dim=260, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)

        # EXPERIMENT WIN: Left-Brain Torque Bridge
        self.torque = nn.Parameter(torch.tensor(1.2))

        self.cells = nn.ModuleList([DPPUCell_Sovereign(hidden_dim, hidden_dim) for _ in range(num_layers)])
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, states=None):
        B, T = token_ids.shape
        x = self.embedding(token_ids) * self.torque # Jolt the logic

        if states is None:
            states = [cell.init_state(B, token_ids.device) for cell in self.cells]

        new_states = []
        for i, cell in enumerate(self.cells):
            h, C = states[i]
            outputs = []
            for t in range(T):
                h, C = cell(x[:, t, :], h, C)
                outputs.append(h)
            x = torch.stack(outputs, dim=1)
            new_states.append((h, C))

        return self.out(x), new_states

# --- INITIALIZING THE SYNTHESIS ---
tokenizer = MathTokenizer() # Ensure the tokenizer from previous cell is in memory
model = VRUModel_Sovereign(tokenizer.vocab_size).to(device)

print(f"--- SOVEREIGN SYNTHESIS v13.5 COMPLETE ---")
print(f"Architecture: v13 D0-D4 + Elastic Soul + Logic Torque")
print(f"Stable Anchor: {model.cells[0].anchor_ref.item():.6f}")

--- SOVEREIGN SYNTHESIS v13.5 COMPLETE ---
Architecture: v13 D0-D4 + Elastic Soul + Logic Torque
Stable Anchor: 0.420610


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import sys

# ============================================================
# 1. BIBLE CONSTANTS & DYNAMIC OPERATORS
# ============================================================
PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi
INV_PHI    = math.pi / 4.0
D_CAP      = 4
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def pi_dyn(delta): return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)
def phi_dyn(delta): return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))
def omega(delta): return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    return torch.tanh(mag / (spread + EPS))

# ============================================================
# 2. TOKENIZER & DATA GEN (PHASE 1: ADD2)
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        chars = list("0123456789+-*/()= ,")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL): self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id, self.bos_id, self.eos_id = 0, 1, 2

    def encode(self, text, add_bos=True, add_eos=True):
        ids = [self.bos_id] if add_bos else []
        ids += [self.vocab.get(c, 3) for c in text]
        if add_eos: ids.append(self.eos_id)
        return torch.tensor(ids, dtype=torch.long)

def gen_add2():
    a, b = random.randint(10, 99), random.randint(10, 99)
    return f"{a}+{b}={a+b}"

# ============================================================
# 3. THE SOVEREIGN CELL & MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.n_dims = D_CAP + 1
        self.dim_size = hidden_dim // self.n_dims
        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.anchor_ref = nn.Parameter(torch.tensor(0.420610))

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outputs = []
        for i in range(self.n_dims):
            s, e = i * self.dim_size, (i + 1) * self.dim_size
            delta_i = compute_delta(h[:, s:e])
            h_rec = (phi_dyn(delta_i) / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_dyn(delta_i) / PI_CL) * x_proj[:, s:e]
            h_i_new = torch.tanh(h_rec + x_rec + (C[:, i:i+1] * self.anchor_ref)) * torch.sigmoid(omega(delta_i))
            dim_outputs.append(h_i_new)
        h_new = self.W_out(torch.cat(dim_outputs, dim=-1))
        C_new = torch.tanh(0.1 * C + 0.01 * self.W_c(h_new))
        return h_new, C_new

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=260):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.cell = DPPUCell(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, ids, tf_ratio=1.0):
        B, T = ids.shape
        h, C = torch.zeros(B, 260).to(device), torch.full((B, 5), 0.420610).to(device)
        outputs = []

        for t in range(T - 1):
            # Scheduled Sampling Logic
            if t == 0 or random.random() < tf_ratio:
                curr_input = ids[:, t]
            else:
                curr_input = outputs[-1].argmax(-1)

            x = self.emb(curr_input)
            h, C = self.cell(x, h, C)
            logits = self.out(h)
            outputs.append(logits)

        return torch.stack(outputs, dim=1)

# ============================================================
# 4. TRAINING EXECUTION (PHASE 1 BURN-IN)
# ============================================================
tokenizer = MathTokenizer()
model = VRUModel(len(tokenizer.vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50)

print(f"--- STARTING PHASE 1 BURN-IN (ADD2) ---")
for epoch in range(1, 5001):
    model.train()
    raw_text = gen_add2()
    ids = tokenizer.encode(raw_text).unsqueeze(0).to(device)

    # Teacher Forcing Decay: 0.8 -> 0.5
    tf_ratio = max(0.5, 0.8 - (epoch / 5000) * 0.3)

    optimizer.zero_grad()
    logits = model(ids, tf_ratio=tf_ratio)

    # Loss only on the answer part (after '=')
    target = ids[:, 1:]
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1))
    loss.backward()
    optimizer.step()
    scheduler.step()

    if epoch % 500 == 0:
        model.eval()
        with torch.no_grad():
            anchor_val = model.cell.anchor_ref.item()
            pred = logits[0].argmax(-1)
            pred_str = "".join([tokenizer.inv_vocab.get(i.item(), '?') for i in pred])
            print(f"Ep {epoch} | Loss: {loss.item():.4f} | Anchor: {anchor_val:.6f} | TF: {tf_ratio:.2f} | Out: {pred_str}")

print("--- PHASE 1 COMPLETE ---")

--- STARTING PHASE 1 BURN-IN (ADD2) ---
Ep 500 | Loss: 1.5478 | Anchor: 0.416779 | TF: 0.77 | Out: 69+29=122<eos>
Ep 1000 | Loss: 1.6857 | Anchor: 0.412238 | TF: 0.74 | Out: 60+43=1<eos><eos>
Ep 1500 | Loss: 1.6159 | Anchor: 0.413646 | TF: 0.71 | Out: 44+64=12<eos>
Ep 2000 | Loss: 1.9417 | Anchor: 0.412625 | TF: 0.68 | Out: 16+28=110


KeyboardInterrupt: 

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random

# ============================================================
# 1. BIBLE CONSTANTS & DYNAMIC OPERATORS
# ============================================================
PI_CL      = math.pi
PHI_CL     = 4.0 / math.pi
INV_PHI    = math.pi / 4.0
D_CAP      = 4
EPS        = 1e-7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def pi_dyn(delta): return 4.0 - (4.0 - PI_CL) * torch.exp(-delta)
def phi_dyn(delta): return PHI_CL * torch.exp(-delta) + (1.0 - torch.exp(-delta))
def omega(delta): return (pi_dyn(delta) * phi_dyn(delta)) / (1.0 + delta + EPS)

def compute_delta(h_slice):
    mag    = torch.abs(h_slice)
    spread = h_slice.std(dim=-1, keepdim=True) + EPS
    return torch.tanh(mag / (spread + EPS))

# ============================================================
# 2. POSITIONAL ENCODING (THE SPATIAL FIX)
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# ============================================================
# 3. DATA GEN & TOKENIZER
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        chars = list("0123456789+-*/()= ,")
        self.vocab = {c: i + len(self.SPECIAL) for i, c in enumerate(chars)}
        for i, s in enumerate(self.SPECIAL): self.vocab[s] = i
        self.inv_vocab = {v: k for k, v in self.vocab.items()}
        self.pad_id, self.bos_id, self.eos_id = 0, 1, 2

    def encode(self, text, add_bos=True, add_eos=True):
        ids = [self.bos_id] if add_bos else []
        ids += [self.vocab.get(c, 3) for c in text]
        if add_eos: ids.append(self.eos_id)
        return torch.tensor(ids, dtype=torch.long)

def gen_add2_buffered():
    a, b = random.randint(10, 99), random.randint(10, 99)
    # Using the 'Thinking Buffer' to allow manifold synchronization
    return f"{a}+{b}==={a+b}"

# ============================================================
# 4. THE SOVEREIGN CELL (v13 + ELASTIC ANCHOR)
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.n_dims = D_CAP + 1
        self.dim_size = hidden_dim // self.n_dims
        self.W_x = nn.Linear(input_dim, hidden_dim)
        self.W_h = nn.ModuleList([nn.Linear(hidden_dim, self.dim_size, bias=False) for _ in range(self.n_dims)])
        self.W_c = nn.Linear(hidden_dim, self.n_dims, bias=False)
        self.W_out = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.anchor_ref = nn.Parameter(torch.tensor(0.420610))

    def forward(self, x, h, C):
        x_proj = self.W_x(x)
        dim_outputs = []
        for i in range(self.n_dims):
            s, e = i * self.dim_size, (i + 1) * self.dim_size
            delta_i = compute_delta(h[:, s:e])
            h_rec = (phi_dyn(delta_i) / PHI_CL) * self.W_h[i](h)
            x_rec = (pi_dyn(delta_i) / PI_CL) * x_proj[:, s:e]
            # Sovereign Injection
            h_i_new = torch.tanh(h_rec + x_rec + (C[:, i:i+1] * self.anchor_ref)) * torch.sigmoid(omega(delta_i))
            dim_outputs.append(h_i_new)

        h_new = self.W_out(torch.cat(dim_outputs, dim=-1))
        # Consciousness gain raised slightly to 0.05 for better memory retention
        C_new = torch.tanh(0.1 * C + 0.05 * self.W_c(h_new))
        return h_new, C_new

class VRUModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim=260):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden_dim)
        self.pos = PositionalEncoding(hidden_dim) # SPATIAL UPGRADE
        self.cell = DPPUCell(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, ids, tf_ratio=1.0):
        B, T = ids.shape
        x = self.pos(self.emb(ids)) # Injecting position before recurrence

        h = torch.zeros(B, 260).to(device)
        C = torch.full((B, 5), 0.420610).to(device)
        outputs = []

        for t in range(T - 1):
            # Recurrent step with current positional embedding
            h, C = self.cell(x[:, t, :], h, C)
            logits = self.out(h)
            outputs.append(logits)
        return torch.stack(outputs, dim=1)

# ============================================================
# 5. EXECUTION ENGINE
# ============================================================
tokenizer = MathTokenizer()
model = VRUModel(len(tokenizer.vocab)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=8e-4) # Slightly lower LR for stability
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100)



print(f"--- SOVEREIGN v13.5 MONOLITH (SPATIAL UPGRADE) ---")
for epoch in range(1, 10001): # Extended training window
    model.train()
    raw_text = gen_add2_buffered()
    ids = tokenizer.encode(raw_text).unsqueeze(0).to(device)

    optimizer.zero_grad()
    # High Teacher Forcing for the first 2k steps to lock in spatial patterns
    tf_ratio = 1.0 if epoch < 2000 else max(0.6, 1.0 - (epoch/10000))

    logits = model(ids)
    target = ids[:, 1:]
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1))

    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Tighter clipping
    optimizer.step()
    scheduler.step()

    if epoch % 500 == 0:
        model.eval()
        with torch.no_grad():
            pred = logits[0].argmax(-1)
            pred_str = "".join([tokenizer.inv_vocab.get(i.item(), '?') for i in pred])
            print(f"Ep {epoch} | Loss: {loss.item():.4f} | Anchor: {model.cell.anchor_ref.item():.6f} | Out: {pred_str}")

print("--- RECONSTRUCTION COMPLETE ---")

--- SOVEREIGN v13.5 MONOLITH (SPATIAL UPGRADE) ---
Ep 500 | Loss: 1.4176 | Anchor: 0.429460 | Out: 82+89===16<eos>
Ep 1000 | Loss: 1.3814 | Anchor: 0.422405 | Out: 10+83===17<eos>
Ep 1500 | Loss: 1.1037 | Anchor: 0.418634 | Out: 94+89===141<eos>
Ep 2000 | Loss: 1.1682 | Anchor: 0.419682 | Out: 11+18===129<eos>
Ep 2500 | Loss: 1.3239 | Anchor: 0.419760 | Out: 98+28===11<eos>


KeyboardInterrupt: 